# Notebook 29 — DINOv2-S Script-Adversarial Projection

## Objective

Notebook 27 established a strong cross-script writer-verification baseline using a frozen DINOv2-S representation and a small trainable writer projection.

Notebook 28 then showed that the writer-trained projection already suppresses a substantial amount of linearly accessible nuisance information without any explicit adversarial objective.

The writer-disjoint script probe changed from:

**1.00000 → 0.75791 ROC-AUC**

and the text-condition probe changed from:

**0.99730 → 0.79244 ROC-AUC**

after the standard writer projection.

However, both nuisance variables remained clearly above chance.

This notebook therefore tests the next controlled hypothesis:

> Can explicit script-adversarial training reduce the remaining linearly accessible script information while preserving the strong cross-script writer-verification performance of the DINOv2-S projection student?

This is not a representation rescue experiment.

The existing DINOv2-S projection student is already a strong baseline. The purpose of this notebook is to test whether a better **writer-discrimination versus script-invariance trade-off** can be obtained.

## Frozen Representation

The backbone representation remains unchanged:

**image → DINOv2-S → cached L2-normalized 384-D feature**

No DINOv2-S transformer block will be fine-tuned in this notebook.

The trainable writer representation remains:

**384-D cached feature → Linear 384→144 → L2 normalization**

This is intentionally matched to the successful Notebook 27 student.

## Adversarial Branch

A script-classification branch will be attached to the 144-D projected writer embedding.

Conceptually:

**DINOv2-S 384-D → writer projection 144-D → writer metric objective**

and simultaneously:

**projected 144-D embedding → gradient reversal → script classifier**

The script classifier predicts:

- Arabic
- English

The script classifier itself learns to identify script correctly.

Through gradient reversal, the writer projection receives the opposite script gradient, encouraging it to make script information less accessible while still satisfying the writer-verification objective.

## Primary Training Objective

The writer objective will remain exactly matched to Notebook 27:

- Arabic within-script writer metric loss
- equal-mean four-condition cross-script writer metric loss
- cosine margin = 0.5
- writer-centric four-page batches

The adversarial experiment therefore changes only one scientific factor:

**explicit script suppression**

No text-condition adversary will be introduced in this notebook.

Text condition will remain a secondary diagnostic so that the effect of script suppression can be interpreted without simultaneously changing two nuisance objectives.

## Clean Development Protocol

The development protocol remains writer-disjoint:

- Writer training set: **145 fit writers**
- Selection set: **36 unseen writers**
- Writer overlap: **0**
- Monitor writers used during method development: **No**
- Validation split used: **No**
- Official test split used: **No**

The current Notebook 27 projection student remains the fixed reference baseline.

Its three-seed cross-script macro AUC was approximately:

**0.83746 ± 0.00185**

and its seed-42 development result was approximately:

**0.83748 macro AUC**

The corresponding seed-42 projected script-probe AUC from Notebook 28 was:

**0.75791**

These references will not be modified after observing the adversarial result.

## Avoiding a Weak-Adversary Failure

Previous ResNet-based adversarial experiments showed that a poorly trained attached script head can appear confused even when script information remains highly accessible to a separate post-hoc probe.

Therefore, low attached-head accuracy or ROC-AUC will not be accepted as evidence of invariance.

Before the main experiment, this notebook will audit:

1. script-head learnability,
2. gradient-reversal sign behavior,
3. writer and adversarial gradient magnitudes,
4. optimizer plumbing,
5. absence of selection leakage.

The adversarial strength will be determined using fit-only optimization diagnostics rather than selection writer-verification outcomes.

After the training protocol is frozen, the true nuisance outcome will be measured using the same independent writer-disjoint post-hoc linear probe protocol used in Notebook 28.

## Evaluation Questions

The experiment will answer three separate questions.

### 1. Writer Verification

Does explicit script suppression preserve the strong cross-script writer-verification performance of the standard DINOv2-S projection student?

### 2. Script Accessibility

Does the independent writer-disjoint script probe fall below the current projected baseline of:

**0.75791 ROC-AUC**

### 3. Trade-off

If script accessibility decreases, how much writer-verification performance is gained, preserved, or lost?

The experiment will therefore be interpreted as a trade-off rather than assuming that stronger invariance must automatically improve verification.

## Important Interpretation Boundary

A lower post-hoc script-probe AUC indicates reduced linearly accessible script information.

It does not prove complete script invariance.

Similarly, a high script-probe AUC does not by itself prove that the verifier causally uses script when making writer decisions.

The main scientific question is whether explicit suppression provides a better representation trade-off than the natural nuisance suppression already learned by the standard writer projection.

## Development Strategy

The experiment will proceed conservatively:

1. audit resources and reproduce the Notebook 27 reference,
2. construct the script-adversarial projection,
3. verify script-head competence,
4. verify adversarial gradient behavior,
5. freeze adversarial strength using fit-only diagnostics,
6. run the controlled seed-42 experiment,
7. compare writer verification and independent nuisance accessibility,
8. run additional predetermined seeds only if the frozen method is scientifically supported.

The untouched monitor set will not be revisited unless a development method passes the predefined selection-stage criteria.

No post-hoc architecture, learning-rate, epoch-budget, or adversarial-strength tuning will be performed after observing the main selection result.

In [1]:
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [2]:
ROOT = Path.cwd().resolve()

if ROOT.name == "notebooks":
    ROOT = ROOT.parent

ROLE_PATH = (
    ROOT
    / "reports"
    / "cross_script_writer_geometry_consistency"
    / "development_internal_writer_roles_seed42.csv"
)

FIT_SCHEDULE_PATH = (
    ROOT
    / "reports"
    / "cross_script_writer_geometry_consistency"
    / "fit_writer_batch_schedule_seed42.csv"
)

SELECTION_PAIR_PATH = (
    ROOT
    / "reports"
    / "cross_script_writer_geometry_consistency"
    / "selection_cross_script_pairs.csv"
)

DINO_S_SOURCE_PATH = (
    ROOT
    / "reports"
    / "dinov2s_student_baseline"
    / "dinov2s_final_refit_aligned_features.npz"
)

BASELINE_CHECKPOINT_PATH = (
    ROOT
    / "checkpoints"
    / "dinov2s_student_baseline"
    / "dinov2s_projection_seed42_best.pt"
)

BASELINE_SEED42_SUMMARY_PATH = (
    ROOT
    / "reports"
    / "dinov2s_student_baseline"
    / "dinov2s_projection_seed42_summary.json"
)

BASELINE_MULTISEED_SUMMARY_PATH = (
    ROOT
    / "reports"
    / "dinov2s_student_baseline"
    / "dinov2s_projection_multiseed_summary.json"
)

BASELINE_MULTISEED_RESULTS_PATH = (
    ROOT
    / "reports"
    / "dinov2s_student_baseline"
    / "dinov2s_projection_multiseed_results.csv"
)

SCRIPT_PROBE_SUMMARY_PATH = (
    ROOT
    / "reports"
    / "dinov2s_student_nuisance_accessibility_audit"
    / "dinov2s_script_probe_summary.json"
)

NUISANCE_VERDICT_PATH = (
    ROOT
    / "reports"
    / "dinov2s_student_nuisance_accessibility_audit"
    / "dinov2s_nuisance_accessibility_verdict.json"
)

FINAL_REFIT_CHECKPOINT_PATH = (
    ROOT
    / "checkpoints"
    / "dinov2s_student_baseline"
    / "dinov2s_projection_final_refit_seed42_epoch10.pt"
)

REPORT_DIR = (
    ROOT
    / "reports"
    / "dinov2s_script_adversarial_projection"
)

CHECKPOINT_DIR = (
    ROOT
    / "checkpoints"
    / "dinov2s_script_adversarial_projection"
)

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

required_paths = {
    "roles": ROLE_PATH,
    "fit_schedule": FIT_SCHEDULE_PATH,
    "selection_pairs": SELECTION_PAIR_PATH,
    "dino_s_source": DINO_S_SOURCE_PATH,
    "baseline_checkpoint": BASELINE_CHECKPOINT_PATH,
    "baseline_seed42_summary": BASELINE_SEED42_SUMMARY_PATH,
    "baseline_multiseed_summary": BASELINE_MULTISEED_SUMMARY_PATH,
    "baseline_multiseed_results": BASELINE_MULTISEED_RESULTS_PATH,
    "script_probe_summary": SCRIPT_PROBE_SUMMARY_PATH,
    "nuisance_verdict": NUISANCE_VERDICT_PATH,
}

missing_paths = {
    name: str(path)
    for name, path in required_paths.items()
    if not path.exists()
}

if len(missing_paths) != 0:
    raise FileNotFoundError(
        json.dumps(
            missing_paths,
            indent=2,
        )
    )

role_df = pd.read_csv(
    ROLE_PATH
)

fit_schedule_df = pd.read_csv(
    FIT_SCHEDULE_PATH
)

selection_pair_df = pd.read_csv(
    SELECTION_PAIR_PATH
)

baseline_multiseed_results_df = pd.read_csv(
    BASELINE_MULTISEED_RESULTS_PATH
)

with open(
    BASELINE_SEED42_SUMMARY_PATH,
    "r",
) as file:
    baseline_seed42_summary = json.load(
        file
    )

with open(
    BASELINE_MULTISEED_SUMMARY_PATH,
    "r",
) as file:
    baseline_multiseed_summary = json.load(
        file
    )

with open(
    SCRIPT_PROBE_SUMMARY_PATH,
    "r",
) as file:
    script_probe_summary = json.load(
        file
    )

with open(
    NUISANCE_VERDICT_PATH,
    "r",
) as file:
    nuisance_verdict = json.load(
        file
    )

baseline_checkpoint = torch.load(
    BASELINE_CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False,
)

role_column = "internal_role"

if role_column not in role_df.columns:
    raise RuntimeError(
        "Expected internal_role column was not found."
    )

fit_role_df = (
    role_df[
        role_df[
            role_column
        ] == "fit"
    ]
    .copy()
)

selection_role_df = (
    role_df[
        role_df[
            role_column
        ] == "selection"
    ]
    .copy()
)

fit_writer_set = set(
    fit_role_df[
        "writer"
    ]
    .astype(int)
    .unique()
)

selection_writer_set = set(
    selection_role_df[
        "writer"
    ]
    .astype(int)
    .unique()
)

schedule_writer_set = set(
    fit_schedule_df[
        "writer"
    ]
    .astype(int)
    .unique()
)

schedule_rows_per_epoch = (
    fit_schedule_df
    .groupby(
        "epoch"
    )
    .size()
)

schedule_batches_per_epoch = (
    fit_schedule_df[
        [
            "epoch",
            "batch",
        ]
    ]
    .drop_duplicates()
    .groupby(
        "epoch"
    )
    .size()
)

schedule_unique_writers_per_epoch = (
    fit_schedule_df
    .groupby(
        "epoch"
    )[
        "writer"
    ]
    .nunique()
)

selection_condition_counts = (
    selection_pair_df[
        "condition"
    ]
    .value_counts()
    .sort_index()
)

BASELINE_SEED42_MACRO_AUC = float(
    baseline_seed42_summary[
        "best_cross_macro_auc"
    ]
)

BASELINE_SEED42_POOLED_AUC = float(
    baseline_seed42_summary[
        "best_pooled_auc"
    ]
)

BASELINE_SEED42_POOLED_EER = float(
    baseline_seed42_summary[
        "best_pooled_eer"
    ]
)

BASELINE_MULTISEED_MACRO_MEAN = float(
    baseline_multiseed_summary[
        "cross_macro_auc_mean"
    ]
)

BASELINE_MULTISEED_MACRO_STD = float(
    baseline_multiseed_summary[
        "cross_macro_auc_std"
    ]
)

BASELINE_SCRIPT_PROBE_AUC = float(
    script_probe_summary[
        "projected_selection_auc"
    ]
)

BASELINE_SCRIPT_PROBE_ACCURACY = float(
    script_probe_summary[
        "projected_selection_accuracy"
    ]
)

BASELINE_TEXT_PROBE_AUC = float(
    nuisance_verdict[
        "projected_text_condition_selection_auc"
    ]
)

BASELINE_RAW_SCRIPT_PROBE_AUC = float(
    script_probe_summary[
        "raw_selection_auc"
    ]
)

baseline_seed_set = (
    baseline_multiseed_results_df[
        "seed"
    ]
    .astype(int)
    .tolist()
)

development_checkpoint_is_final_refit = bool(
    BASELINE_CHECKPOINT_PATH.resolve()
    == FINAL_REFIT_CHECKPOINT_PATH.resolve()
)

notebook29_resource_protocol_audit = {
    "notebook": 29,
    "experiment": (
        "DINOv2-S script-adversarial projection"
    ),
    "fit_pages": int(
        len(
            fit_role_df
        )
    ),
    "fit_writers": int(
        len(
            fit_writer_set
        )
    ),
    "selection_pages": int(
        len(
            selection_role_df
        )
    ),
    "selection_writers": int(
        len(
            selection_writer_set
        )
    ),
    "fit_selection_writer_overlap": int(
        len(
            fit_writer_set
            & selection_writer_set
        )
    ),
    "schedule_epochs": int(
        fit_schedule_df[
            "epoch"
        ].nunique()
    ),
    "schedule_rows": int(
        len(
            fit_schedule_df
        )
    ),
    "schedule_writers": int(
        len(
            schedule_writer_set
        )
    ),
    "all_epochs_have_145_writer_rows": bool(
        (
            schedule_rows_per_epoch
            == 145
        ).all()
    ),
    "all_epochs_have_18_batches": bool(
        (
            schedule_batches_per_epoch
            == 18
        ).all()
    ),
    "all_epochs_have_145_unique_writers": bool(
        (
            schedule_unique_writers_per_epoch
            == 145
        ).all()
    ),
    "schedule_matches_fit_writers": bool(
        schedule_writer_set
        == fit_writer_set
    ),
    "selection_pairs": int(
        len(
            selection_pair_df
        )
    ),
    "selection_conditions": int(
        selection_pair_df[
            "condition"
        ].nunique()
    ),
    "selection_pairs_per_condition": {
        str(key): int(value)
        for key, value in (
            selection_condition_counts
            .to_dict()
            .items()
        )
    },
    "baseline_checkpoint_epoch": int(
        baseline_checkpoint[
            "epoch"
        ]
    ),
    "baseline_checkpoint_training_seed": int(
        baseline_checkpoint[
            "training_seed"
        ]
    ),
    "baseline_projection_dimension": int(
        baseline_checkpoint[
            "projection_dimension"
        ]
    ),
    "baseline_checkpoint_fit_writers": int(
        baseline_checkpoint[
            "fit_writers"
        ]
    ),
    "baseline_checkpoint_selection_writers": int(
        baseline_checkpoint[
            "selection_writers"
        ]
    ),
    "baseline_checkpoint_source_features_l2_normalized": bool(
        baseline_checkpoint[
            "source_features_l2_normalized"
        ]
    ),
    "development_checkpoint_is_final_refit_checkpoint": bool(
        development_checkpoint_is_final_refit
    ),
    "baseline_seed42_cross_macro_auc": float(
        BASELINE_SEED42_MACRO_AUC
    ),
    "baseline_seed42_pooled_auc": float(
        BASELINE_SEED42_POOLED_AUC
    ),
    "baseline_seed42_pooled_eer": float(
        BASELINE_SEED42_POOLED_EER
    ),
    "baseline_multiseed_seeds": [
        int(seed)
        for seed in baseline_seed_set
    ],
    "baseline_multiseed_cross_macro_auc_mean": float(
        BASELINE_MULTISEED_MACRO_MEAN
    ),
    "baseline_multiseed_cross_macro_auc_std": float(
        BASELINE_MULTISEED_MACRO_STD
    ),
    "raw_dinov2s_script_probe_auc": float(
        BASELINE_RAW_SCRIPT_PROBE_AUC
    ),
    "baseline_projected_script_probe_auc": float(
        BASELINE_SCRIPT_PROBE_AUC
    ),
    "baseline_projected_script_probe_accuracy": float(
        BASELINE_SCRIPT_PROBE_ACCURACY
    ),
    "baseline_projected_text_condition_probe_auc": float(
        BASELINE_TEXT_PROBE_AUC
    ),
    "baseline_reference_locked": True,
    "writer_objective_will_match_notebook27": True,
    "script_is_only_adversarial_target": True,
    "text_condition_is_diagnostic_only": True,
    "adversarial_strength_will_use_fit_only_diagnostics": True,
    "selection_will_not_set_adversarial_strength": True,
    "final_refit_checkpoint_will_not_be_used_for_development": True,
    "monitor_used": False,
    "validation_used": False,
    "official_test_used": False,
    "adversarial_training_started": False,
    "selection_evaluated_in_this_notebook": False,
}

with open(
    REPORT_DIR
    / "dinov2s_script_adversarial_resource_protocol_audit.json",
    "w",
) as file:
    json.dump(
        notebook29_resource_protocol_audit,
        file,
        indent=2,
    )

print(
    json.dumps(
        notebook29_resource_protocol_audit,
        indent=2,
    )
)

if (
    len(
        fit_role_df
    ) != 580
    or len(
        fit_writer_set
    ) != 145
    or len(
        selection_role_df
    ) != 144
    or len(
        selection_writer_set
    ) != 36
    or len(
        fit_writer_set
        & selection_writer_set
    ) != 0
    or fit_schedule_df[
        "epoch"
    ].nunique()
    != 10
    or len(
        fit_schedule_df
    ) != 1450
    or not (
        schedule_rows_per_epoch
        == 145
    ).all()
    or not (
        schedule_batches_per_epoch
        == 18
    ).all()
    or not (
        schedule_unique_writers_per_epoch
        == 145
    ).all()
    or schedule_writer_set
    != fit_writer_set
    or len(
        selection_pair_df
    ) != 5184
    or selection_pair_df[
        "condition"
    ].nunique()
    != 4
    or not (
        selection_condition_counts
        == 1296
    ).all()
    or int(
        baseline_checkpoint[
            "epoch"
        ]
    ) != 10
    or int(
        baseline_checkpoint[
            "training_seed"
        ]
    ) != 42
    or int(
        baseline_checkpoint[
            "projection_dimension"
        ]
    ) != 144
    or int(
        baseline_checkpoint[
            "fit_writers"
        ]
    ) != 145
    or int(
        baseline_checkpoint[
            "selection_writers"
        ]
    ) != 36
    or not baseline_checkpoint[
        "source_features_l2_normalized"
    ]
    or development_checkpoint_is_final_refit
    or set(
        baseline_seed_set
    )
    != {
        42,
        123,
        2026,
    }
    or not np.isclose(
        BASELINE_SEED42_MACRO_AUC,
        0.8374834656084656,
        rtol=0.0,
        atol=1e-12,
    )
    or not np.isclose(
        BASELINE_MULTISEED_MACRO_MEAN,
        0.8374632569077014,
        rtol=0.0,
        atol=1e-12,
    )
    or not np.isclose(
        BASELINE_SCRIPT_PROBE_AUC,
        0.757908950617284,
        rtol=0.0,
        atol=1e-12,
    )
):
    raise RuntimeError(
        "Notebook 29 resource/protocol audit failed."
    )

{
  "notebook": 29,
  "experiment": "DINOv2-S script-adversarial projection",
  "fit_pages": 580,
  "fit_writers": 145,
  "selection_pages": 144,
  "selection_writers": 36,
  "fit_selection_writer_overlap": 0,
  "schedule_epochs": 10,
  "schedule_rows": 1450,
  "schedule_writers": 145,
  "all_epochs_have_145_writer_rows": true,
  "all_epochs_have_18_batches": true,
  "all_epochs_have_145_unique_writers": true,
  "schedule_matches_fit_writers": true,
  "selection_pairs": 5184,
  "selection_conditions": 4,
  "selection_pairs_per_condition": {
    "cross_same_same": 1296,
    "cross_same_variable": 1296,
    "cross_variable_same": 1296,
    "cross_variable_variable": 1296
  },
  "baseline_checkpoint_epoch": 10,
  "baseline_checkpoint_training_seed": 42,
  "baseline_projection_dimension": 144,
  "baseline_checkpoint_fit_writers": 145,
  "baseline_checkpoint_selection_writers": 36,
  "baseline_checkpoint_source_features_l2_normalized": true,
  "development_checkpoint_is_final_refit_checkpoi

In [3]:
DINO_S_FEATURE_DIM = 384
PROJECTION_DIM = 144
COSINE_MARGIN = 0.5

source_archive = np.load(
    DINO_S_SOURCE_PATH,
    allow_pickle=False,
)

required_source_keys = {
    "refit_embeddings",
    "refit_filenames",
    "refit_writers",
    "refit_page_ids",
}

if not required_source_keys.issubset(
    source_archive.files
):
    raise RuntimeError(
        "DINOv2-S source artifact is incomplete."
    )

source_embeddings = (
    source_archive[
        "refit_embeddings"
    ]
    .astype(
        np.float32,
        copy=False,
    )
)

source_filenames = (
    source_archive[
        "refit_filenames"
    ]
    .astype(str)
)

source_writers = (
    source_archive[
        "refit_writers"
    ]
    .astype(
        np.int64,
        copy=False,
    )
)

source_page_ids = (
    source_archive[
        "refit_page_ids"
    ]
    .astype(
        np.int64,
        copy=False,
    )
)

source_metadata_df = pd.DataFrame(
    {
        "filename": source_filenames,
        "source_writer": source_writers,
        "page_id": source_page_ids,
        "source_index": np.arange(
            len(
                source_filenames
            ),
            dtype=np.int64,
        ),
    }
)

development_role_df = (
    role_df[
        role_df[
            role_column
        ].isin(
            [
                "fit",
                "selection",
            ]
        )
    ][
        [
            "filename",
            "writer",
            role_column,
        ]
    ]
    .copy()
)

development_role_df[
    "filename"
] = (
    development_role_df[
        "filename"
    ]
    .astype(str)
)

aligned_metadata_df = (
    development_role_df
    .merge(
        source_metadata_df,
        on="filename",
        how="inner",
        validate="one_to_one",
    )
)

if not (
    aligned_metadata_df[
        "writer"
    ]
    .astype(int)
    .to_numpy()
    ==
    aligned_metadata_df[
        "source_writer"
    ]
    .astype(int)
    .to_numpy()
).all():
    raise RuntimeError(
        "Writer mismatch during DINOv2-S source alignment."
    )

aligned_metadata_df[
    "writer"
] = (
    aligned_metadata_df[
        "writer"
    ]
    .astype(int)
)

aligned_metadata_df[
    "page_id"
] = (
    aligned_metadata_df[
        "page_id"
    ]
    .astype(int)
)

aligned_metadata_df[
    "script_label"
] = np.where(
    aligned_metadata_df[
        "page_id"
    ].isin(
        [
            3,
            4,
        ]
    ),
    1,
    0,
).astype(
    np.int64
)

fit_rows = (
    aligned_metadata_df[
        aligned_metadata_df[
            role_column
        ] == "fit"
    ]
    .sort_values(
        [
            "writer",
            "page_id",
        ]
    )
    .reset_index(
        drop=True
    )
)

selection_rows = (
    aligned_metadata_df[
        aligned_metadata_df[
            role_column
        ] == "selection"
    ]
    .sort_values(
        [
            "writer",
            "page_id",
        ]
    )
    .reset_index(
        drop=True
    )
)

fit_raw_features = (
    source_embeddings[
        fit_rows[
            "source_index"
        ]
        .astype(int)
        .to_numpy()
    ]
)

selection_raw_features = (
    source_embeddings[
        selection_rows[
            "source_index"
        ]
        .astype(int)
        .to_numpy()
    ]
)

fit_raw_features = (
    fit_raw_features
    .astype(
        np.float32,
        copy=False,
    )
)

selection_raw_features = (
    selection_raw_features
    .astype(
        np.float32,
        copy=False,
    )
)

fit_writer_feature_map = {}

for writer, writer_rows in (
    fit_rows
    .groupby(
        "writer",
        sort=True,
    )
):
    writer_rows = (
        writer_rows
        .sort_values(
            "page_id"
        )
    )

    page_ids = (
        writer_rows[
            "page_id"
        ]
        .astype(int)
        .tolist()
    )

    indices = (
        writer_rows[
            "source_index"
        ]
        .astype(int)
        .to_numpy()
    )

    writer_features = (
        source_embeddings[
            indices
        ]
        .astype(
            np.float32,
            copy=False,
        )
    )

    if (
        page_ids
        != [
            1,
            2,
            3,
            4,
        ]
        or writer_features.shape
        != (
            4,
            DINO_S_FEATURE_DIM,
        )
    ):
        raise RuntimeError(
            f"Unexpected fit writer structure for writer {writer}."
        )

    fit_writer_feature_map[
        int(
            writer
        )
    ] = (
        writer_features
        .copy()
    )

fit_writer_ids = set(
    fit_writer_feature_map.keys()
)

selection_writer_ids = set(
    selection_rows[
        "writer"
    ]
    .astype(int)
    .unique()
)

fit_script_labels = (
    fit_rows[
        "script_label"
    ]
    .to_numpy(
        dtype=np.int64
    )
)

selection_script_labels = (
    selection_rows[
        "script_label"
    ]
    .to_numpy(
        dtype=np.int64
    )
)


class DINOv2SProjectionStudent(
    nn.Module
):
    def __init__(
        self,
        input_dimension=384,
        projection_dimension=144,
    ):
        super().__init__()

        self.projection = nn.Linear(
            input_dimension,
            projection_dimension,
            bias=True,
        )

    def forward(
        self,
        features,
    ):
        features = F.normalize(
            features,
            p=2,
            dim=-1,
        )

        embeddings = (
            self.projection(
                features
            )
        )

        embeddings = F.normalize(
            embeddings,
            p=2,
            dim=-1,
        )

        return embeddings


baseline_projection_model = (
    DINOv2SProjectionStudent(
        input_dimension=(
            DINO_S_FEATURE_DIM
        ),
        projection_dimension=(
            PROJECTION_DIM
        ),
    )
)

baseline_projection_model.load_state_dict(
    baseline_checkpoint[
        "model_state_dict"
    ],
    strict=True,
)

baseline_projection_model.eval()

with torch.no_grad():
    baseline_fit_embeddings = (
        baseline_projection_model(
            torch.from_numpy(
                fit_raw_features
            ).float()
        )
        .cpu()
        .numpy()
    )

    baseline_selection_embeddings = (
        baseline_projection_model(
            torch.from_numpy(
                selection_raw_features
            ).float()
        )
        .cpu()
        .numpy()
    )

fit_raw_norms = np.linalg.norm(
    fit_raw_features,
    axis=1,
)

selection_raw_norms = np.linalg.norm(
    selection_raw_features,
    axis=1,
)

baseline_fit_embedding_norms = np.linalg.norm(
    baseline_fit_embeddings,
    axis=1,
)

baseline_selection_embedding_norms = np.linalg.norm(
    baseline_selection_embeddings,
    axis=1,
)


def calculate_interpolated_eer(
    labels,
    scores,
):
    fpr, tpr, thresholds = roc_curve(
        labels,
        scores,
    )

    fnr = (
        1.0
        - tpr
    )

    difference = (
        fpr
        - fnr
    )

    crossing_indices = np.where(
        np.diff(
            np.sign(
                difference
            )
        )
        != 0
    )[0]

    if len(
        crossing_indices
    ) == 0:
        nearest_index = int(
            np.argmin(
                np.abs(
                    difference
                )
            )
        )

        eer = (
            fpr[
                nearest_index
            ]
            + fnr[
                nearest_index
            ]
        ) / 2.0

        threshold = (
            thresholds[
                nearest_index
            ]
        )

        return (
            float(
                eer
            ),
            float(
                threshold
            ),
        )

    index = int(
        crossing_indices[
            0
        ]
    )

    difference_left = (
        difference[
            index
        ]
    )

    difference_right = (
        difference[
            index
            + 1
        ]
    )

    interpolation_weight = (
        -difference_left
        / (
            difference_right
            - difference_left
        )
    )

    eer = (
        fpr[
            index
        ]
        + interpolation_weight
        * (
            fpr[
                index
                + 1
            ]
            - fpr[
                index
            ]
        )
    )

    threshold = (
        thresholds[
            index
        ]
        + interpolation_weight
        * (
            thresholds[
                index
                + 1
            ]
            - thresholds[
                index
            ]
        )
    )

    return (
        float(
            eer
        ),
        float(
            threshold
        ),
    )


def evaluate_selection_embedding_map(
    embedding_map,
):
    left_embeddings = np.stack(
        [
            embedding_map[
                filename
            ]
            for filename in (
                selection_pair_df[
                    "left_filename"
                ]
                .astype(str)
            )
        ]
    )

    right_embeddings = np.stack(
        [
            embedding_map[
                filename
            ]
            for filename in (
                selection_pair_df[
                    "right_filename"
                ]
                .astype(str)
            )
        ]
    )

    labels = (
        selection_pair_df[
            "label"
        ]
        .to_numpy(
            dtype=np.int64
        )
    )

    scores = np.sum(
        left_embeddings
        * right_embeddings,
        axis=1,
    )

    pooled_auc = float(
        roc_auc_score(
            labels,
            scores,
        )
    )

    pooled_eer, pooled_threshold = (
        calculate_interpolated_eer(
            labels,
            scores,
        )
    )

    scored_pairs_df = (
        selection_pair_df
        .copy()
    )

    scored_pairs_df[
        "score"
    ] = scores

    condition_rows = []

    for condition in sorted(
        scored_pairs_df[
            "condition"
        ].unique()
    ):
        condition_df = (
            scored_pairs_df[
                scored_pairs_df[
                    "condition"
                ] == condition
            ]
        )

        condition_labels = (
            condition_df[
                "label"
            ]
            .to_numpy(
                dtype=np.int64
            )
        )

        condition_scores = (
            condition_df[
                "score"
            ]
            .to_numpy()
        )

        condition_auc = float(
            roc_auc_score(
                condition_labels,
                condition_scores,
            )
        )

        condition_eer, _ = (
            calculate_interpolated_eer(
                condition_labels,
                condition_scores,
            )
        )

        condition_rows.append(
            {
                "condition": (
                    condition
                ),
                "auc": float(
                    condition_auc
                ),
                "eer": float(
                    condition_eer
                ),
                "pairs": int(
                    len(
                        condition_df
                    )
                ),
                "genuine_pairs": int(
                    condition_df[
                        "label"
                    ].sum()
                ),
                "impostor_pairs": int(
                    (
                        condition_df[
                            "label"
                        ] == 0
                    ).sum()
                ),
            }
        )

    condition_df = pd.DataFrame(
        condition_rows
    )

    return {
        "cross_macro_auc": float(
            condition_df[
                "auc"
            ].mean()
        ),
        "cross_macro_eer": float(
            condition_df[
                "eer"
            ].mean()
        ),
        "pooled_auc": float(
            pooled_auc
        ),
        "pooled_eer": float(
            pooled_eer
        ),
        "pooled_eer_threshold": float(
            pooled_threshold
        ),
        "condition_df": (
            condition_df
        ),
    }


baseline_selection_embedding_map = {
    filename: embedding
    for filename, embedding in zip(
        selection_rows[
            "filename"
        ].astype(str),
        baseline_selection_embeddings,
    )
}

baseline_reproduction = (
    evaluate_selection_embedding_map(
        baseline_selection_embedding_map
    )
)

baseline_macro_difference = float(
    baseline_reproduction[
        "cross_macro_auc"
    ]
    - BASELINE_SEED42_MACRO_AUC
)

baseline_pooled_auc_difference = float(
    baseline_reproduction[
        "pooled_auc"
    ]
    - BASELINE_SEED42_POOLED_AUC
)

baseline_pooled_eer_difference = float(
    baseline_reproduction[
        "pooled_eer"
    ]
    - BASELINE_SEED42_POOLED_EER
)

baseline_condition_df = (
    baseline_reproduction[
        "condition_df"
    ]
)

ALIGNED_FEATURE_PATH = (
    REPORT_DIR
    / "dinov2s_script_adversarial_fit_selection_features.npz"
)

np.savez_compressed(
    ALIGNED_FEATURE_PATH,
    fit_raw_features=(
        fit_raw_features
    ),
    selection_raw_features=(
        selection_raw_features
    ),
    baseline_fit_embeddings=(
        baseline_fit_embeddings
    ),
    baseline_selection_embeddings=(
        baseline_selection_embeddings
    ),
    fit_filenames=np.asarray(
        fit_rows[
            "filename"
        ]
        .astype(str)
        .tolist(),
        dtype=str,
    ),
    selection_filenames=np.asarray(
        selection_rows[
            "filename"
        ]
        .astype(str)
        .tolist(),
        dtype=str,
    ),
    fit_writers=np.asarray(
        fit_rows[
            "writer"
        ]
        .astype(int)
        .to_numpy(),
        dtype=np.int64,
    ),
    selection_writers=np.asarray(
        selection_rows[
            "writer"
        ]
        .astype(int)
        .to_numpy(),
        dtype=np.int64,
    ),
    fit_page_ids=np.asarray(
        fit_rows[
            "page_id"
        ]
        .astype(int)
        .to_numpy(),
        dtype=np.int64,
    ),
    selection_page_ids=np.asarray(
        selection_rows[
            "page_id"
        ]
        .astype(int)
        .to_numpy(),
        dtype=np.int64,
    ),
    fit_script_labels=(
        fit_script_labels
    ),
    selection_script_labels=(
        selection_script_labels
    ),
)

baseline_condition_df.to_csv(
    REPORT_DIR
    / "notebook27_seed42_baseline_reproduction_conditions.csv",
    index=False,
)

notebook29_feature_baseline_audit = {
    "source_pages": int(
        len(
            source_embeddings
        )
    ),
    "source_writers": int(
        len(
            np.unique(
                source_writers
            )
        )
    ),
    "fit_pages": int(
        len(
            fit_rows
        )
    ),
    "fit_writers": int(
        len(
            fit_writer_ids
        )
    ),
    "selection_pages": int(
        len(
            selection_rows
        )
    ),
    "selection_writers": int(
        len(
            selection_writer_ids
        )
    ),
    "fit_selection_writer_overlap": int(
        len(
            fit_writer_ids
            & selection_writer_ids
        )
    ),
    "fit_raw_shape": list(
        fit_raw_features.shape
    ),
    "selection_raw_shape": list(
        selection_raw_features.shape
    ),
    "baseline_fit_embedding_shape": list(
        baseline_fit_embeddings.shape
    ),
    "baseline_selection_embedding_shape": list(
        baseline_selection_embeddings.shape
    ),
    "raw_features_unit_normalized": bool(
        np.allclose(
            fit_raw_norms,
            1.0,
            rtol=0.0,
            atol=1e-5,
        )
        and np.allclose(
            selection_raw_norms,
            1.0,
            rtol=0.0,
            atol=1e-5,
        )
    ),
    "baseline_embeddings_unit_normalized": bool(
        np.allclose(
            baseline_fit_embedding_norms,
            1.0,
            rtol=0.0,
            atol=1e-5,
        )
        and np.allclose(
            baseline_selection_embedding_norms,
            1.0,
            rtol=0.0,
            atol=1e-5,
        )
    ),
    "fit_script_counts": {
        "Arabic": int(
            (
                fit_script_labels
                == 0
            ).sum()
        ),
        "English": int(
            (
                fit_script_labels
                == 1
            ).sum()
        ),
    },
    "selection_script_counts": {
        "Arabic": int(
            (
                selection_script_labels
                == 0
            ).sum()
        ),
        "English": int(
            (
                selection_script_labels
                == 1
            ).sum()
        ),
    },
    "baseline_expected_cross_macro_auc": float(
        BASELINE_SEED42_MACRO_AUC
    ),
    "baseline_reproduced_cross_macro_auc": float(
        baseline_reproduction[
            "cross_macro_auc"
        ]
    ),
    "baseline_cross_macro_auc_difference": float(
        baseline_macro_difference
    ),
    "baseline_expected_pooled_auc": float(
        BASELINE_SEED42_POOLED_AUC
    ),
    "baseline_reproduced_pooled_auc": float(
        baseline_reproduction[
            "pooled_auc"
        ]
    ),
    "baseline_pooled_auc_difference": float(
        baseline_pooled_auc_difference
    ),
    "baseline_expected_pooled_eer": float(
        BASELINE_SEED42_POOLED_EER
    ),
    "baseline_reproduced_pooled_eer": float(
        baseline_reproduction[
            "pooled_eer"
        ]
    ),
    "baseline_pooled_eer_difference": float(
        baseline_pooled_eer_difference
    ),
    "selection_pair_count": int(
        len(
            selection_pair_df
        )
    ),
    "conditions": int(
        len(
            baseline_condition_df
        )
    ),
    "condition_pair_counts_valid": bool(
        (
            baseline_condition_df[
                "pairs"
            ]
            == 1296
        ).all()
    ),
    "fit_writer_feature_map_ready": bool(
        len(
            fit_writer_feature_map
        ) == 145
    ),
    "aligned_feature_artifact": str(
        ALIGNED_FEATURE_PATH.relative_to(
            ROOT
        )
    ),
    "baseline_evaluator_reproduced_exactly": bool(
        abs(
            baseline_macro_difference
        )
        <= 1e-12
        and abs(
            baseline_pooled_auc_difference
        )
        <= 1e-12
        and abs(
            baseline_pooled_eer_difference
        )
        <= 1e-12
    ),
    "adversarial_model_created": False,
    "adversarial_training_started": False,
    "adversarial_selection_evaluated": False,
    "monitor_used": False,
    "validation_used": False,
    "official_test_used": False,
}

with open(
    REPORT_DIR
    / "dinov2s_script_adversarial_feature_baseline_audit.json",
    "w",
) as file:
    json.dump(
        notebook29_feature_baseline_audit,
        file,
        indent=2,
    )

print(
    json.dumps(
        notebook29_feature_baseline_audit,
        indent=2,
    )
)

print(
    "\nNotebook 27 seed-42 baseline condition reproduction:"
)

print(
    baseline_condition_df
    .round(
        6
    )
    .to_string(
        index=False
    )
)

if (
    source_embeddings.shape
    != (
        724,
        384,
    )
    or fit_raw_features.shape
    != (
        580,
        384,
    )
    or selection_raw_features.shape
    != (
        144,
        384,
    )
    or baseline_fit_embeddings.shape
    != (
        580,
        144,
    )
    or baseline_selection_embeddings.shape
    != (
        144,
        144,
    )
    or len(
        fit_writer_feature_map
    ) != 145
    or len(
        fit_writer_ids
        & selection_writer_ids
    ) != 0
    or not notebook29_feature_baseline_audit[
        "raw_features_unit_normalized"
    ]
    or not notebook29_feature_baseline_audit[
        "baseline_embeddings_unit_normalized"
    ]
    or notebook29_feature_baseline_audit[
        "fit_script_counts"
    ]
    != {
        "Arabic": 290,
        "English": 290,
    }
    or notebook29_feature_baseline_audit[
        "selection_script_counts"
    ]
    != {
        "Arabic": 72,
        "English": 72,
    }
    or not notebook29_feature_baseline_audit[
        "condition_pair_counts_valid"
    ]
    or not notebook29_feature_baseline_audit[
        "baseline_evaluator_reproduced_exactly"
    ]
):
    raise RuntimeError(
        "Notebook 29 feature/baseline audit failed."
    )

{
  "source_pages": 724,
  "source_writers": 181,
  "fit_pages": 580,
  "fit_writers": 145,
  "selection_pages": 144,
  "selection_writers": 36,
  "fit_selection_writer_overlap": 0,
  "fit_raw_shape": [
    580,
    384
  ],
  "selection_raw_shape": [
    144,
    384
  ],
  "baseline_fit_embedding_shape": [
    580,
    144
  ],
  "baseline_selection_embedding_shape": [
    144,
    144
  ],
  "raw_features_unit_normalized": true,
  "baseline_embeddings_unit_normalized": true,
  "fit_script_counts": {
    "Arabic": 290,
    "English": 290
  },
  "selection_script_counts": {
    "Arabic": 72,
    "English": 72
  },
  "baseline_expected_cross_macro_auc": 0.8374834656084656,
  "baseline_reproduced_cross_macro_auc": 0.8374834656084656,
  "baseline_cross_macro_auc_difference": 0.0,
  "baseline_expected_pooled_auc": 0.8372946979717814,
  "baseline_reproduced_pooled_auc": 0.8372946979717814,
  "baseline_pooled_auc_difference": 0.0,
  "baseline_expected_pooled_eer": 0.24285714285714285,
  "ba

In [4]:
ADVERSARIAL_SEED = 42
SCRIPT_CLASSES = 2
SCRIPT_GRL_SANITY_LAMBDA = 1.0


def set_experiment_seed(
    seed,
):
    random.seed(
        seed
    )

    np.random.seed(
        seed
    )

    torch.manual_seed(
        seed
    )


class GradientReversalFunction(
    torch.autograd.Function
):
    @staticmethod
    def forward(
        ctx,
        inputs,
        strength,
    ):
        ctx.strength = float(
            strength
        )

        return inputs.view_as(
            inputs
        )

    @staticmethod
    def backward(
        ctx,
        gradient_output,
    ):
        return (
            -ctx.strength
            * gradient_output,
            None,
        )


def gradient_reverse(
    inputs,
    strength,
):
    return GradientReversalFunction.apply(
        inputs,
        strength,
    )


class DINOv2SScriptAdversarialStudent(
    nn.Module
):
    def __init__(
        self,
        input_dimension=384,
        projection_dimension=144,
        script_classes=2,
    ):
        super().__init__()

        self.projection = nn.Linear(
            input_dimension,
            projection_dimension,
            bias=True,
        )

        nn.init.xavier_uniform_(
            self.projection.weight
        )

        nn.init.zeros_(
            self.projection.bias
        )

        self.script_head = nn.Linear(
            projection_dimension,
            script_classes,
            bias=True,
        )

        nn.init.xavier_uniform_(
            self.script_head.weight
        )

        nn.init.zeros_(
            self.script_head.bias
        )

    def project(
        self,
        features,
    ):
        features = F.normalize(
            features,
            p=2,
            dim=-1,
        )

        embeddings = self.projection(
            features
        )

        embeddings = F.normalize(
            embeddings,
            p=2,
            dim=-1,
        )

        return embeddings

    def classify_script(
        self,
        embeddings,
        grl_strength=1.0,
        use_grl=True,
    ):
        if use_grl:
            classifier_input = gradient_reverse(
                embeddings,
                grl_strength,
            )
        else:
            classifier_input = embeddings

        return self.script_head(
            classifier_input
        )

    def forward(
        self,
        features,
        grl_strength=1.0,
        use_grl=True,
    ):
        embeddings = self.project(
            features
        )

        script_logits = self.classify_script(
            embeddings,
            grl_strength=grl_strength,
            use_grl=use_grl,
        )

        return {
            "embeddings": embeddings,
            "script_logits": script_logits,
        }


def balanced_writer_metric_loss(
    left_embeddings,
    right_embeddings,
    margin=COSINE_MARGIN,
):
    similarity_matrix = (
        left_embeddings
        @ right_embeddings.T
    )

    batch_size = int(
        similarity_matrix.shape[
            0
        ]
    )

    positive_mask = torch.eye(
        batch_size,
        dtype=torch.bool,
        device=similarity_matrix.device,
    )

    negative_mask = ~positive_mask

    positive_similarities = (
        similarity_matrix[
            positive_mask
        ]
    )

    negative_similarities = (
        similarity_matrix[
            negative_mask
        ]
    )

    positive_loss = (
        1.0
        - positive_similarities
    ).mean()

    negative_loss = torch.clamp(
        negative_similarities
        - margin,
        min=0.0,
    ).mean()

    loss = (
        positive_loss
        + negative_loss
    ) / 2.0

    return {
        "loss": loss,
        "positive_loss": positive_loss,
        "negative_loss": negative_loss,
        "positive_similarity_mean": (
            positive_similarities.mean()
        ),
        "negative_similarity_mean": (
            negative_similarities.mean()
        ),
    }


CROSS_SCRIPT_PAGE_PAIRS = {
    "cross_variable_variable": (
        0,
        2,
    ),
    "cross_variable_same": (
        0,
        3,
    ),
    "cross_same_variable": (
        1,
        2,
    ),
    "cross_same_same": (
        1,
        3,
    ),
}


def build_writer_objective(
    embeddings,
):
    arabic_result = (
        balanced_writer_metric_loss(
            embeddings[
                :,
                0,
                :,
            ],
            embeddings[
                :,
                1,
                :,
            ],
        )
    )

    cross_results = {}

    for condition, (
        left_page,
        right_page,
    ) in (
        CROSS_SCRIPT_PAGE_PAIRS
        .items()
    ):
        cross_results[
            condition
        ] = (
            balanced_writer_metric_loss(
                embeddings[
                    :,
                    left_page,
                    :,
                ],
                embeddings[
                    :,
                    right_page,
                    :,
                ],
            )
        )

    cross_metric_loss = torch.stack(
        [
            result[
                "loss"
            ]
            for result in (
                cross_results.values()
            )
        ]
    ).mean()

    base_loss = (
        arabic_result[
            "loss"
        ]
        + cross_metric_loss
    )

    return {
        "base_loss": base_loss,
        "arabic_result": arabic_result,
        "cross_metric_loss": (
            cross_metric_loss
        ),
        "cross_results": (
            cross_results
        ),
    }


first_schedule_batch = (
    fit_schedule_df[
        (
            fit_schedule_df[
                "epoch"
            ] == 1
        )
        & (
            fit_schedule_df[
                "batch"
            ] == 1
        )
    ]
    .sort_values(
        "slot"
    )
)

first_batch_writers = (
    first_schedule_batch[
        "writer"
    ]
    .astype(int)
    .tolist()
)

first_batch_features = np.stack(
    [
        fit_writer_feature_map[
            writer
        ]
        for writer in (
            first_batch_writers
        )
    ],
    axis=0,
).astype(
    np.float32,
    copy=False,
)

first_batch_tensor = (
    torch.from_numpy(
        first_batch_features
    )
    .float()
)

first_batch_script_labels = (
    torch.tensor(
        [
            0,
            0,
            1,
            1,
        ],
        dtype=torch.long,
    )
    .unsqueeze(
        0
    )
    .repeat(
        first_batch_tensor.shape[
            0
        ],
        1,
    )
)


set_experiment_seed(
    ADVERSARIAL_SEED
)

adversarial_sanity_model = (
    DINOv2SScriptAdversarialStudent(
        input_dimension=(
            DINO_S_FEATURE_DIM
        ),
        projection_dimension=(
            PROJECTION_DIM
        ),
        script_classes=(
            SCRIPT_CLASSES
        ),
    )
)

adversarial_sanity_model.eval()

with torch.no_grad():
    sanity_output = (
        adversarial_sanity_model(
            first_batch_tensor,
            grl_strength=(
                SCRIPT_GRL_SANITY_LAMBDA
            ),
            use_grl=True,
        )
    )

sanity_embeddings = (
    sanity_output[
        "embeddings"
    ]
)

sanity_script_logits = (
    sanity_output[
        "script_logits"
    ]
)

sanity_writer_objective = (
    build_writer_objective(
        sanity_embeddings
    )
)

sanity_script_loss = (
    F.cross_entropy(
        sanity_script_logits.reshape(
            -1,
            SCRIPT_CLASSES,
        ),
        first_batch_script_labels.reshape(
            -1
        ),
    )
)

sanity_script_predictions = (
    sanity_script_logits
    .argmax(
        dim=-1
    )
)

sanity_script_accuracy = float(
    (
        sanity_script_predictions
        == first_batch_script_labels
    )
    .float()
    .mean()
    .item()
)

sanity_embedding_norms = torch.linalg.vector_norm(
    sanity_embeddings,
    dim=-1,
)

projection_parameter_count = int(
    sum(
        parameter.numel()
        for parameter in (
            adversarial_sanity_model
            .projection
            .parameters()
        )
    )
)

script_head_parameter_count = int(
    sum(
        parameter.numel()
        for parameter in (
            adversarial_sanity_model
            .script_head
            .parameters()
        )
    )
)

total_parameter_count = int(
    sum(
        parameter.numel()
        for parameter in (
            adversarial_sanity_model
            .parameters()
        )
    )
)

expected_notebook27_initial_base_loss = (
    0.45000821352005005
)

initial_base_loss_difference = float(
    sanity_writer_objective[
        "base_loss"
    ]
    .item()
    - expected_notebook27_initial_base_loss
)

condition_sanity_rows = []

for condition, result in (
    sanity_writer_objective[
        "cross_results"
    ]
    .items()
):
    condition_sanity_rows.append(
        {
            "condition": (
                condition
            ),
            "loss": float(
                result[
                    "loss"
                ].item()
            ),
            "positive_similarity_mean": float(
                result[
                    "positive_similarity_mean"
                ].item()
            ),
            "negative_similarity_mean": float(
                result[
                    "negative_similarity_mean"
                ].item()
            ),
        }
    )

adversarial_forward_condition_df = (
    pd.DataFrame(
        condition_sanity_rows
    )
)

adversarial_architecture_forward_audit = {
    "method": (
        "DINOv2-S script-adversarial projection"
    ),
    "seed": int(
        ADVERSARIAL_SEED
    ),
    "input_dimension": int(
        DINO_S_FEATURE_DIM
    ),
    "projection_dimension": int(
        PROJECTION_DIM
    ),
    "script_classes": int(
        SCRIPT_CLASSES
    ),
    "script_head_architecture": (
        "Linear(144, 2)"
    ),
    "script_head_matches_linear_accessibility_target": True,
    "projection_parameters": int(
        projection_parameter_count
    ),
    "script_head_parameters": int(
        script_head_parameter_count
    ),
    "total_trainable_parameters": int(
        total_parameter_count
    ),
    "first_batch_writers": (
        first_batch_writers
    ),
    "first_batch_size": int(
        first_batch_tensor.shape[
            0
        ]
    ),
    "first_batch_input_shape": list(
        first_batch_tensor.shape
    ),
    "embedding_shape": list(
        sanity_embeddings.shape
    ),
    "script_logits_shape": list(
        sanity_script_logits.shape
    ),
    "script_label_shape": list(
        first_batch_script_labels.shape
    ),
    "script_label_counts": {
        "Arabic": int(
            (
                first_batch_script_labels
                == 0
            ).sum()
            .item()
        ),
        "English": int(
            (
                first_batch_script_labels
                == 1
            ).sum()
            .item()
        ),
    },
    "embedding_norm_min": float(
        sanity_embedding_norms
        .min()
        .item()
    ),
    "embedding_norm_mean": float(
        sanity_embedding_norms
        .mean()
        .item()
    ),
    "embedding_norm_max": float(
        sanity_embedding_norms
        .max()
        .item()
    ),
    "initial_writer_base_loss": float(
        sanity_writer_objective[
            "base_loss"
        ].item()
    ),
    "expected_notebook27_initial_writer_base_loss": float(
        expected_notebook27_initial_base_loss
    ),
    "initial_writer_base_loss_difference": float(
        initial_base_loss_difference
    ),
    "writer_objective_initialization_matches_notebook27": bool(
        abs(
            initial_base_loss_difference
        )
        <= 1e-7
    ),
    "initial_script_cross_entropy": float(
        sanity_script_loss.item()
    ),
    "initial_script_accuracy": float(
        sanity_script_accuracy
    ),
    "grl_structurally_present": True,
    "grl_strength_used_for_forward_sanity": float(
        SCRIPT_GRL_SANITY_LAMBDA
    ),
    "grl_gradient_sign_verified": False,
    "script_head_competence_verified": False,
    "adversarial_strength_frozen": False,
    "main_training_started": False,
    "selection_evaluated": False,
    "monitor_used": False,
    "validation_used": False,
    "official_test_used": False,
}

adversarial_forward_condition_df.to_csv(
    REPORT_DIR
    / "dinov2s_script_adversarial_initial_condition_sanity.csv",
    index=False,
)

with open(
    REPORT_DIR
    / "dinov2s_script_adversarial_architecture_forward_audit.json",
    "w",
) as file:
    json.dump(
        adversarial_architecture_forward_audit,
        file,
        indent=2,
    )

print(
    json.dumps(
        adversarial_architecture_forward_audit,
        indent=2,
    )
)

print(
    "\nInitial cross-script condition sanity:"
)

print(
    adversarial_forward_condition_df
    .round(
        6
    )
    .to_string(
        index=False
    )
)

if (
    projection_parameter_count
    != 55440
    or script_head_parameter_count
    != 290
    or total_parameter_count
    != 55730
    or first_batch_tensor.shape
    != (
        8,
        4,
        384,
    )
    or sanity_embeddings.shape
    != (
        8,
        4,
        144,
    )
    or sanity_script_logits.shape
    != (
        8,
        4,
        2,
    )
    or not torch.allclose(
        sanity_embedding_norms,
        torch.ones_like(
            sanity_embedding_norms
        ),
        rtol=0.0,
        atol=1e-5,
    )
    or abs(
        initial_base_loss_difference
    )
    > 1e-7
):
    raise RuntimeError(
        "Notebook 29 adversarial architecture forward audit failed."
    )

{
  "method": "DINOv2-S script-adversarial projection",
  "seed": 42,
  "input_dimension": 384,
  "projection_dimension": 144,
  "script_classes": 2,
  "script_head_architecture": "Linear(144, 2)",
  "script_head_matches_linear_accessibility_target": true,
  "projection_parameters": 55440,
  "script_head_parameters": 290,
  "total_trainable_parameters": 55730,
  "first_batch_writers": [
    133,
    39,
    273,
    138,
    172,
    249,
    272,
    94
  ],
  "first_batch_size": 8,
  "first_batch_input_shape": [
    8,
    4,
    384
  ],
  "embedding_shape": [
    8,
    4,
    144
  ],
  "script_logits_shape": [
    8,
    4,
    2
  ],
  "script_label_shape": [
    8,
    4
  ],
  "script_label_counts": {
    "Arabic": 16,
    "English": 16
  },
  "embedding_norm_min": 0.9999998807907104,
  "embedding_norm_mean": 1.0,
  "embedding_norm_max": 1.0,
  "initial_writer_base_loss": 0.45000821352005005,
  "expected_notebook27_initial_writer_base_loss": 0.45000821352005005,
  "initial_wri

In [6]:
SCRIPT_HEAD_WARMUP_EPOCHS = 10
SCRIPT_HEAD_LR = 1e-3
SCRIPT_HEAD_WEIGHT_DECAY = 1e-4
SCRIPT_HEAD_COMPETENCE_AUC_THRESHOLD = 0.90
SCRIPT_HEAD_COMPETENCE_ACCURACY_THRESHOLD = 0.80


def load_fit_writer_batch(
    epoch,
    batch_id,
):
    batch_schedule = (
        fit_schedule_df[
            (
                fit_schedule_df[
                    "epoch"
                ] == epoch
            )
            & (
                fit_schedule_df[
                    "batch"
                ] == batch_id
            )
        ]
        .sort_values(
            "slot"
        )
    )

    writers = (
        batch_schedule[
            "writer"
        ]
        .astype(int)
        .tolist()
    )

    features = np.stack(
        [
            fit_writer_feature_map[
                writer
            ]
            for writer in writers
        ],
        axis=0,
    ).astype(
        np.float32,
        copy=False,
    )

    labels = (
        torch.tensor(
            [
                0,
                0,
                1,
                1,
            ],
            dtype=torch.long,
        )
        .unsqueeze(
            0
        )
        .repeat(
            len(
                writers
            ),
            1,
        )
    )

    return {
        "writers": writers,
        "features": features,
        "script_labels": labels,
    }


set_experiment_seed(
    ADVERSARIAL_SEED
)

script_head_warmup_model = (
    DINOv2SScriptAdversarialStudent(
        input_dimension=(
            DINO_S_FEATURE_DIM
        ),
        projection_dimension=(
            PROJECTION_DIM
        ),
        script_classes=(
            SCRIPT_CLASSES
        ),
    )
)

for parameter in (
    script_head_warmup_model
    .projection
    .parameters()
):
    parameter.requires_grad = False

projection_before_warmup = {
    name: parameter
    .detach()
    .clone()
    for name, parameter in (
        script_head_warmup_model
        .projection
        .named_parameters()
    )
}

script_head_before_warmup = {
    name: parameter
    .detach()
    .clone()
    for name, parameter in (
        script_head_warmup_model
        .script_head
        .named_parameters()
    )
}

script_head_optimizer = torch.optim.AdamW(
    script_head_warmup_model
    .script_head
    .parameters(),
    lr=SCRIPT_HEAD_LR,
    weight_decay=(
        SCRIPT_HEAD_WEIGHT_DECAY
    ),
)

script_head_warmup_model.eval()

with torch.no_grad():
    initial_fit_embeddings = (
        script_head_warmup_model
        .project(
            torch.from_numpy(
                fit_raw_features
            ).float()
        )
    )

    initial_fit_logits = (
        script_head_warmup_model
        .classify_script(
            initial_fit_embeddings,
            use_grl=False,
        )
    )

    initial_full_fit_script_loss = float(
        F.cross_entropy(
            initial_fit_logits,
            torch.from_numpy(
                fit_script_labels
            ).long(),
        )
        .item()
    )

warmup_history_rows = []

for epoch in range(
    1,
    SCRIPT_HEAD_WARMUP_EPOCHS
    + 1,
):
    epoch_batch_ids = (
        fit_schedule_df[
            fit_schedule_df[
                "epoch"
            ] == epoch
        ][
            "batch"
        ]
        .drop_duplicates()
        .sort_values()
        .astype(int)
        .tolist()
    )

    epoch_losses = []
    epoch_correct = 0
    epoch_examples = 0

    script_head_warmup_model.train()

    for batch_id in (
        epoch_batch_ids
    ):
        batch = (
            load_fit_writer_batch(
                epoch=epoch,
                batch_id=batch_id,
            )
        )

        batch_features = (
            torch.from_numpy(
                batch[
                    "features"
                ]
            )
            .float()
        )

        batch_labels = (
            batch[
                "script_labels"
            ]
        )

        with torch.no_grad():
            batch_embeddings = (
                script_head_warmup_model
                .project(
                    batch_features
                )
            )

        script_head_optimizer.zero_grad(
            set_to_none=True
        )

        script_logits = (
            script_head_warmup_model
            .classify_script(
                batch_embeddings,
                use_grl=False,
            )
        )

        script_loss = (
            F.cross_entropy(
                script_logits.reshape(
                    -1,
                    SCRIPT_CLASSES,
                ),
                batch_labels.reshape(
                    -1
                ),
            )
        )

        script_loss.backward()

        script_head_optimizer.step()

        predictions = (
            script_logits
            .argmax(
                dim=-1
            )
        )

        epoch_losses.append(
            float(
                script_loss
                .detach()
                .item()
            )
        )

        epoch_correct += int(
            (
                predictions
                == batch_labels
            )
            .sum()
            .item()
        )

        epoch_examples += int(
            batch_labels.numel()
        )

    warmup_history_rows.append(
        {
            "epoch": int(
                epoch
            ),
            "mean_script_loss": float(
                np.mean(
                    epoch_losses
                )
            ),
            "training_accuracy": float(
                epoch_correct
                / epoch_examples
            ),
            "examples_seen": int(
                epoch_examples
            ),
            "batches": int(
                len(
                    epoch_batch_ids
                )
            ),
        }
    )

    print(
        f"script-head-warmup | "
        f"epoch {epoch:02d} | "
        f"loss "
        f"{np.mean(epoch_losses):.6f} | "
        f"accuracy "
        f"{epoch_correct / epoch_examples:.4f}"
    )


script_head_warmup_history_df = pd.DataFrame(
    warmup_history_rows
)

script_head_warmup_model.eval()

with torch.no_grad():
    warmed_fit_embeddings = (
        script_head_warmup_model
        .project(
            torch.from_numpy(
                fit_raw_features
            ).float()
        )
    )

    warmed_fit_logits = (
        script_head_warmup_model
        .classify_script(
            warmed_fit_embeddings,
            use_grl=False,
        )
    )

    warmed_fit_probabilities = (
        torch.softmax(
            warmed_fit_logits,
            dim=1,
        )[
            :,
            1,
        ]
        .cpu()
        .numpy()
    )

    warmed_fit_predictions = (
        warmed_fit_logits
        .argmax(
            dim=1
        )
        .cpu()
        .numpy()
    )

    final_full_fit_script_loss = float(
        F.cross_entropy(
            warmed_fit_logits,
            torch.from_numpy(
                fit_script_labels
            ).long(),
        )
        .item()
    )


final_fit_script_auc = float(
    roc_auc_score(
        fit_script_labels,
        warmed_fit_probabilities,
    )
)

final_fit_script_accuracy = float(
    accuracy_score(
        fit_script_labels,
        warmed_fit_predictions,
    )
)

final_fit_script_balanced_accuracy = float(
    balanced_accuracy_score(
        fit_script_labels,
        warmed_fit_predictions,
    )
)

final_fit_script_confusion = (
    confusion_matrix(
        fit_script_labels,
        warmed_fit_predictions,
        labels=[
            0,
            1,
        ],
    )
)


projection_update_squared_sum = 0.0
projection_maximum_absolute_update = 0.0

for name, parameter in (
    script_head_warmup_model
    .projection
    .named_parameters()
):
    difference = (
        parameter
        .detach()
        - projection_before_warmup[
            name
        ]
    )

    projection_update_squared_sum += float(
        difference
        .double()
        .pow(
            2
        )
        .sum()
        .item()
    )

    projection_maximum_absolute_update = max(
        projection_maximum_absolute_update,
        float(
            difference
            .abs()
            .max()
            .item()
        ),
    )

projection_update_norm = float(
    np.sqrt(
        projection_update_squared_sum
    )
)


script_head_update_squared_sum = 0.0
script_head_maximum_absolute_update = 0.0

for name, parameter in (
    script_head_warmup_model
    .script_head
    .named_parameters()
):
    difference = (
        parameter
        .detach()
        - script_head_before_warmup[
            name
        ]
    )

    script_head_update_squared_sum += float(
        difference
        .double()
        .pow(
            2
        )
        .sum()
        .item()
    )

    script_head_maximum_absolute_update = max(
        script_head_maximum_absolute_update,
        float(
            difference
            .abs()
            .max()
            .item()
        ),
    )

script_head_update_norm = float(
    np.sqrt(
        script_head_update_squared_sum
    )
)


first_batch_after_warmup = (
    load_fit_writer_batch(
        epoch=1,
        batch_id=1,
    )
)

with torch.no_grad():
    first_batch_after_warmup_embeddings = (
        script_head_warmup_model
        .project(
            torch.from_numpy(
                first_batch_after_warmup[
                    "features"
                ]
            ).float()
        )
    )

    first_batch_after_warmup_writer_objective = (
        build_writer_objective(
            first_batch_after_warmup_embeddings
        )
    )

writer_loss_after_head_warmup = float(
    first_batch_after_warmup_writer_objective[
        "base_loss"
    ]
    .item()
)

writer_loss_change_after_head_warmup = float(
    writer_loss_after_head_warmup
    - expected_notebook27_initial_base_loss
)


script_head_competence_verified = bool(
    final_fit_script_auc
    >= SCRIPT_HEAD_COMPETENCE_AUC_THRESHOLD
    and final_fit_script_accuracy
    >= SCRIPT_HEAD_COMPETENCE_ACCURACY_THRESHOLD
    and final_full_fit_script_loss
    < initial_full_fit_script_loss
    and projection_update_norm
    == 0.0
)


SCRIPT_HEAD_WARMUP_PATH = (
    CHECKPOINT_DIR
    / "fit_only_script_head_warmup_seed42.pt"
)

torch.save(
    {
        "training_stage": (
            "fit_only_script_head_competence_warmup"
        ),
        "seed": int(
            ADVERSARIAL_SEED
        ),
        "epochs": int(
            SCRIPT_HEAD_WARMUP_EPOCHS
        ),
        "learning_rate": float(
            SCRIPT_HEAD_LR
        ),
        "weight_decay": float(
            SCRIPT_HEAD_WEIGHT_DECAY
        ),
        "script_head_state_dict": (
            script_head_warmup_model
            .script_head
            .state_dict()
        ),
        "projection_state_dict": (
            script_head_warmup_model
            .projection
            .state_dict()
        ),
        "fit_script_auc": float(
            final_fit_script_auc
        ),
        "fit_script_accuracy": float(
            final_fit_script_accuracy
        ),
        "selection_used": False,
        "monitor_used": False,
        "validation_used": False,
        "official_test_used": False,
    },
    SCRIPT_HEAD_WARMUP_PATH,
)


script_head_warmup_history_df.to_csv(
    REPORT_DIR
    / "fit_only_script_head_warmup_history.csv",
    index=False,
)


script_head_competence_audit = {
    "purpose": (
        "verify attached script-head competence "
        "without using selection"
    ),
    "projection_initialization": (
        "same seed-42 Xavier initialization as Notebook 27"
    ),
    "projection_frozen_during_warmup": True,
    "script_head_architecture": (
        "Linear(144, 2)"
    ),
    "warmup_epochs": int(
        SCRIPT_HEAD_WARMUP_EPOCHS
    ),
    "script_head_learning_rate": float(
        SCRIPT_HEAD_LR
    ),
    "script_head_weight_decay": float(
        SCRIPT_HEAD_WEIGHT_DECAY
    ),
    "fit_writers": 145,
    "fit_pages": 580,
    "fit_script_counts": {
        "Arabic": int(
            (
                fit_script_labels
                == 0
            ).sum()
        ),
        "English": int(
            (
                fit_script_labels
                == 1
            ).sum()
        ),
    },
    "initial_full_fit_script_loss": float(
        initial_full_fit_script_loss
    ),
    "final_full_fit_script_loss": float(
        final_full_fit_script_loss
    ),
    "final_fit_script_auc": float(
        final_fit_script_auc
    ),
    "final_fit_script_accuracy": float(
        final_fit_script_accuracy
    ),
    "final_fit_script_balanced_accuracy": float(
        final_fit_script_balanced_accuracy
    ),
    "final_fit_script_confusion_matrix": (
        final_fit_script_confusion
        .astype(int)
        .tolist()
    ),
    "competence_auc_threshold": float(
        SCRIPT_HEAD_COMPETENCE_AUC_THRESHOLD
    ),
    "competence_accuracy_threshold": float(
        SCRIPT_HEAD_COMPETENCE_ACCURACY_THRESHOLD
    ),
    "projection_parameter_update_norm": float(
        projection_update_norm
    ),
    "projection_maximum_absolute_update": float(
        projection_maximum_absolute_update
    ),
    "script_head_parameter_update_norm": float(
        script_head_update_norm
    ),
    "script_head_maximum_absolute_update": float(
        script_head_maximum_absolute_update
    ),
    "first_batch_writer_loss_after_head_warmup": float(
        writer_loss_after_head_warmup
    ),
    "writer_loss_change_after_head_warmup": float(
        writer_loss_change_after_head_warmup
    ),
    "writer_projection_preserved_exactly": bool(
        projection_update_norm
        == 0.0
        and abs(
            writer_loss_change_after_head_warmup
        )
        <= 1e-7
    ),
    "script_head_competence_verified": bool(
        script_head_competence_verified
    ),
    "warm_head_will_initialize_main_adversary_if_competence_passes": True,
    "selection_used": False,
    "selection_evaluated": False,
    "monitor_used": False,
    "validation_used": False,
    "official_test_used": False,
    "main_adversarial_training_started": False,
}


with open(
    REPORT_DIR
    / "fit_only_script_head_competence_audit.json",
    "w",
) as file:
    json.dump(
        script_head_competence_audit,
        file,
        indent=2,
    )


print(
    "\nFit-only script-head competence audit:"
)

print(
    json.dumps(
        script_head_competence_audit,
        indent=2,
    )
)

print(
    "\nWarm-up history:"
)

print(
    script_head_warmup_history_df
    .round(
        6
    )
    .to_string(
        index=False
    )
)


if not script_head_competence_verified:
    raise RuntimeError(
        "Attached script head did not pass the fit-only competence audit."
    )

/tmp/ipykernel_68722/1967841766.py:159: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  torch.from_numpy(


script-head-warmup | epoch 01 | loss 0.692994 | accuracy 0.5000
script-head-warmup | epoch 02 | loss 0.678610 | accuracy 0.6345
script-head-warmup | epoch 03 | loss 0.665207 | accuracy 0.7103
script-head-warmup | epoch 04 | loss 0.652671 | accuracy 0.7552
script-head-warmup | epoch 05 | loss 0.640572 | accuracy 0.7690
script-head-warmup | epoch 06 | loss 0.628775 | accuracy 0.7914
script-head-warmup | epoch 07 | loss 0.617693 | accuracy 0.8155
script-head-warmup | epoch 08 | loss 0.606774 | accuracy 0.8276
script-head-warmup | epoch 09 | loss 0.596509 | accuracy 0.8397
script-head-warmup | epoch 10 | loss 0.586432 | accuracy 0.8448

Fit-only script-head competence audit:
{
  "purpose": "verify attached script-head competence without using selection",
  "projection_initialization": "same seed-42 Xavier initialization as Notebook 27",
  "projection_frozen_during_warmup": true,
  "script_head_architecture": "Linear(144, 2)",
  "warmup_epochs": 10,
  "script_head_learning_rate": 0.001,
  "

In [7]:
direct_gradient_model = (
    DINOv2SScriptAdversarialStudent(
        input_dimension=DINO_S_FEATURE_DIM,
        projection_dimension=PROJECTION_DIM,
        script_classes=SCRIPT_CLASSES,
    )
)

reversed_gradient_model = (
    DINOv2SScriptAdversarialStudent(
        input_dimension=DINO_S_FEATURE_DIM,
        projection_dimension=PROJECTION_DIM,
        script_classes=SCRIPT_CLASSES,
    )
)

direct_gradient_model.load_state_dict(
    script_head_warmup_model.state_dict(),
    strict=True,
)

reversed_gradient_model.load_state_dict(
    script_head_warmup_model.state_dict(),
    strict=True,
)

direct_gradient_model.zero_grad(
    set_to_none=True
)

direct_output = (
    direct_gradient_model(
        first_batch_tensor,
        grl_strength=1.0,
        use_grl=False,
    )
)

direct_script_loss = (
    F.cross_entropy(
        direct_output[
            "script_logits"
        ].reshape(
            -1,
            SCRIPT_CLASSES,
        ),
        first_batch_script_labels.reshape(
            -1
        ),
    )
)

direct_script_loss.backward()


reversed_gradient_model.zero_grad(
    set_to_none=True
)

reversed_output = (
    reversed_gradient_model(
        first_batch_tensor,
        grl_strength=1.0,
        use_grl=True,
    )
)

reversed_script_loss = (
    F.cross_entropy(
        reversed_output[
            "script_logits"
        ].reshape(
            -1,
            SCRIPT_CLASSES,
        ),
        first_batch_script_labels.reshape(
            -1
        ),
    )
)

reversed_script_loss.backward()


direct_projection_gradients = []

reversed_projection_gradients = []

direct_head_gradients = []

reversed_head_gradients = []

gradient_rows = []


for parameter_name in [
    "weight",
    "bias",
]:
    direct_gradient = (
        getattr(
            direct_gradient_model.projection,
            parameter_name,
        )
        .grad
        .detach()
        .clone()
    )

    reversed_gradient = (
        getattr(
            reversed_gradient_model.projection,
            parameter_name,
        )
        .grad
        .detach()
        .clone()
    )

    direct_projection_gradients.append(
        direct_gradient.reshape(
            -1
        )
    )

    reversed_projection_gradients.append(
        reversed_gradient.reshape(
            -1
        )
    )

    gradient_rows.append(
        {
            "parameter_group": (
                "projection"
            ),
            "parameter": (
                parameter_name
            ),
            "direct_gradient_norm": float(
                direct_gradient.norm().item()
            ),
            "reversed_gradient_norm": float(
                reversed_gradient.norm().item()
            ),
            "cosine_similarity": float(
                F.cosine_similarity(
                    direct_gradient.reshape(
                        1,
                        -1,
                    ),
                    reversed_gradient.reshape(
                        1,
                        -1,
                    ),
                    dim=1,
                ).item()
            ),
            "maximum_absolute_sum": float(
                (
                    direct_gradient
                    + reversed_gradient
                )
                .abs()
                .max()
                .item()
            ),
        }
    )


for parameter_name in [
    "weight",
    "bias",
]:
    direct_gradient = (
        getattr(
            direct_gradient_model.script_head,
            parameter_name,
        )
        .grad
        .detach()
        .clone()
    )

    reversed_gradient = (
        getattr(
            reversed_gradient_model.script_head,
            parameter_name,
        )
        .grad
        .detach()
        .clone()
    )

    direct_head_gradients.append(
        direct_gradient.reshape(
            -1
        )
    )

    reversed_head_gradients.append(
        reversed_gradient.reshape(
            -1
        )
    )

    gradient_rows.append(
        {
            "parameter_group": (
                "script_head"
            ),
            "parameter": (
                parameter_name
            ),
            "direct_gradient_norm": float(
                direct_gradient.norm().item()
            ),
            "reversed_gradient_norm": float(
                reversed_gradient.norm().item()
            ),
            "cosine_similarity": float(
                F.cosine_similarity(
                    direct_gradient.reshape(
                        1,
                        -1,
                    ),
                    reversed_gradient.reshape(
                        1,
                        -1,
                    ),
                    dim=1,
                ).item()
            ),
            "maximum_absolute_sum": float(
                (
                    direct_gradient
                    - reversed_gradient
                )
                .abs()
                .max()
                .item()
            ),
        }
    )


direct_projection_gradient_vector = (
    torch.cat(
        direct_projection_gradients
    )
)

reversed_projection_gradient_vector = (
    torch.cat(
        reversed_projection_gradients
    )
)

direct_head_gradient_vector = (
    torch.cat(
        direct_head_gradients
    )
)

reversed_head_gradient_vector = (
    torch.cat(
        reversed_head_gradients
    )
)


projection_gradient_cosine = float(
    F.cosine_similarity(
        direct_projection_gradient_vector.reshape(
            1,
            -1,
        ),
        reversed_projection_gradient_vector.reshape(
            1,
            -1,
        ),
        dim=1,
    ).item()
)

script_head_gradient_cosine = float(
    F.cosine_similarity(
        direct_head_gradient_vector.reshape(
            1,
            -1,
        ),
        reversed_head_gradient_vector.reshape(
            1,
            -1,
        ),
        dim=1,
    ).item()
)

direct_projection_gradient_norm = float(
    direct_projection_gradient_vector.norm().item()
)

reversed_projection_gradient_norm = float(
    reversed_projection_gradient_vector.norm().item()
)

direct_head_gradient_norm = float(
    direct_head_gradient_vector.norm().item()
)

reversed_head_gradient_norm = float(
    reversed_head_gradient_vector.norm().item()
)


projection_negative_residual = float(
    (
        direct_projection_gradient_vector
        + reversed_projection_gradient_vector
    )
    .abs()
    .max()
    .item()
)

head_equal_residual = float(
    (
        direct_head_gradient_vector
        - reversed_head_gradient_vector
    )
    .abs()
    .max()
    .item()
)


forward_embedding_difference = float(
    (
        direct_output[
            "embeddings"
        ]
        - reversed_output[
            "embeddings"
        ]
    )
    .abs()
    .max()
    .item()
)

forward_logit_difference = float(
    (
        direct_output[
            "script_logits"
        ]
        - reversed_output[
            "script_logits"
        ]
    )
    .abs()
    .max()
    .item()
)

script_loss_difference = float(
    abs(
        direct_script_loss.item()
        - reversed_script_loss.item()
    )
)


grl_gradient_parameter_df = (
    pd.DataFrame(
        gradient_rows
    )
)


grl_gradient_sign_verified = bool(
    direct_projection_gradient_norm
    > 0.0
    and reversed_projection_gradient_norm
    > 0.0
    and direct_head_gradient_norm
    > 0.0
    and reversed_head_gradient_norm
    > 0.0
    and projection_gradient_cosine
    < -0.999999
    and script_head_gradient_cosine
    > 0.999999
    and projection_negative_residual
    <= 1e-7
    and head_equal_residual
    <= 1e-7
    and forward_embedding_difference
    == 0.0
    and forward_logit_difference
    == 0.0
    and script_loss_difference
    == 0.0
)


grl_gradient_sign_audit = {
    "purpose": (
        "verify that GRL reverses only the projection-side "
        "script gradient while preserving the script-head gradient"
    ),
    "script_head_initialization": (
        "fit-only competent warmed head"
    ),
    "grl_strength": 1.0,
    "direct_script_loss": float(
        direct_script_loss.item()
    ),
    "reversed_script_loss": float(
        reversed_script_loss.item()
    ),
    "script_loss_difference": float(
        script_loss_difference
    ),
    "forward_embedding_maximum_absolute_difference": float(
        forward_embedding_difference
    ),
    "forward_script_logit_maximum_absolute_difference": float(
        forward_logit_difference
    ),
    "direct_projection_gradient_norm": float(
        direct_projection_gradient_norm
    ),
    "reversed_projection_gradient_norm": float(
        reversed_projection_gradient_norm
    ),
    "projection_gradient_norm_ratio_reversed_over_direct": float(
        reversed_projection_gradient_norm
        / direct_projection_gradient_norm
    ),
    "projection_direct_vs_reversed_cosine": float(
        projection_gradient_cosine
    ),
    "projection_direct_plus_reversed_maximum_absolute_residual": float(
        projection_negative_residual
    ),
    "direct_script_head_gradient_norm": float(
        direct_head_gradient_norm
    ),
    "reversed_script_head_gradient_norm": float(
        reversed_head_gradient_norm
    ),
    "script_head_gradient_norm_ratio_reversed_over_direct": float(
        reversed_head_gradient_norm
        / direct_head_gradient_norm
    ),
    "script_head_direct_vs_reversed_cosine": float(
        script_head_gradient_cosine
    ),
    "script_head_direct_minus_reversed_maximum_absolute_residual": float(
        head_equal_residual
    ),
    "projection_gradient_reversed": bool(
        projection_gradient_cosine
        < -0.999999
    ),
    "script_head_gradient_preserved": bool(
        script_head_gradient_cosine
        > 0.999999
    ),
    "grl_gradient_sign_verified": bool(
        grl_gradient_sign_verified
    ),
    "optimizer_step_performed": False,
    "parameters_updated": False,
    "selection_used": False,
    "selection_evaluated": False,
    "monitor_used": False,
    "validation_used": False,
    "official_test_used": False,
    "main_adversarial_training_started": False,
}


grl_gradient_parameter_df.to_csv(
    REPORT_DIR
    / "grl_gradient_sign_parameter_audit.csv",
    index=False,
)

with open(
    REPORT_DIR
    / "grl_gradient_sign_audit.json",
    "w",
) as file:
    json.dump(
        grl_gradient_sign_audit,
        file,
        indent=2,
    )


print(
    json.dumps(
        grl_gradient_sign_audit,
        indent=2,
    )
)

print(
    "\nGRL parameter-gradient audit:"
)

print(
    grl_gradient_parameter_df
    .round(
        9
    )
    .to_string(
        index=False
    )
)


if not grl_gradient_sign_verified:
    raise RuntimeError(
        "Gradient-reversal sign audit failed."
    )

{
  "purpose": "verify that GRL reverses only the projection-side script gradient while preserving the script-head gradient",
  "script_head_initialization": "fit-only competent warmed head",
  "grl_strength": 1.0,
  "direct_script_loss": 0.5931294560432434,
  "reversed_script_loss": 0.5931294560432434,
  "script_loss_difference": 0.0,
  "forward_embedding_maximum_absolute_difference": 0.0,
  "forward_script_logit_maximum_absolute_difference": 0.0,
  "direct_projection_gradient_norm": 0.2435365915298462,
  "reversed_projection_gradient_norm": 0.2435365915298462,
  "projection_gradient_norm_ratio_reversed_over_direct": 1.0,
  "projection_direct_vs_reversed_cosine": -1.000001311302185,
  "projection_direct_plus_reversed_maximum_absolute_residual": 0.0,
  "direct_script_head_gradient_norm": 0.05421948805451393,
  "reversed_script_head_gradient_norm": 0.05421948805451393,
  "script_head_gradient_norm_ratio_reversed_over_direct": 1.0,
  "script_head_direct_vs_reversed_cosine": 1.00000011920

In [8]:
GRADIENT_AUDIT_EPOCH = 1
GRADIENT_AUDIT_BATCH_IDS = [
    1,
    2,
    3,
    4,
    5,
    6,
]

TARGET_ADVERSARIAL_TO_WRITER_GRAD_RATIO = 0.20

MIN_GRL_STRENGTH = 0.05
MAX_GRL_STRENGTH = 4.0


def get_projection_gradient_vector(
    model,
):
    gradients = []

    for parameter in (
        model.projection.parameters()
    ):
        if parameter.grad is None:
            raise RuntimeError(
                "Missing projection gradient."
            )

        gradients.append(
            parameter.grad
            .detach()
            .reshape(
                -1
            )
            .clone()
        )

    return torch.cat(
        gradients
    )


gradient_balance_rows = []


for batch_id in (
    GRADIENT_AUDIT_BATCH_IDS
):
    audit_model = (
        DINOv2SScriptAdversarialStudent(
            input_dimension=(
                DINO_S_FEATURE_DIM
            ),
            projection_dimension=(
                PROJECTION_DIM
            ),
            script_classes=(
                SCRIPT_CLASSES
            ),
        )
    )

    audit_model.load_state_dict(
        script_head_warmup_model.state_dict(),
        strict=True,
    )

    for parameter in (
        audit_model.script_head.parameters()
    ):
        parameter.requires_grad = False

    audit_model.train()

    batch = (
        load_fit_writer_batch(
            epoch=(
                GRADIENT_AUDIT_EPOCH
            ),
            batch_id=batch_id,
        )
    )

    batch_features = (
        torch.from_numpy(
            batch[
                "features"
            ]
        )
        .float()
    )

    batch_script_labels = (
        batch[
            "script_labels"
        ]
    )

    audit_model.zero_grad(
        set_to_none=True
    )

    writer_embeddings = (
        audit_model.project(
            batch_features
        )
    )

    writer_objective = (
        build_writer_objective(
            writer_embeddings
        )
    )

    writer_loss = (
        writer_objective[
            "base_loss"
        ]
    )

    writer_loss.backward()

    writer_gradient_vector = (
        get_projection_gradient_vector(
            audit_model
        )
    )

    writer_gradient_norm = float(
        writer_gradient_vector
        .norm()
        .item()
    )

    audit_model.zero_grad(
        set_to_none=True
    )

    adversarial_output = (
        audit_model(
            batch_features,
            grl_strength=1.0,
            use_grl=True,
        )
    )

    script_loss = (
        F.cross_entropy(
            adversarial_output[
                "script_logits"
            ]
            .reshape(
                -1,
                SCRIPT_CLASSES,
            ),
            batch_script_labels
            .reshape(
                -1
            ),
        )
    )

    script_loss.backward()

    adversarial_gradient_vector = (
        get_projection_gradient_vector(
            audit_model
        )
    )

    adversarial_gradient_norm = float(
        adversarial_gradient_vector
        .norm()
        .item()
    )

    raw_gradient_ratio = float(
        adversarial_gradient_norm
        / writer_gradient_norm
    )

    writer_vs_adversarial_cosine = float(
        F.cosine_similarity(
            writer_gradient_vector.reshape(
                1,
                -1,
            ),
            adversarial_gradient_vector.reshape(
                1,
                -1,
            ),
            dim=1,
        )
        .item()
    )

    gradient_balance_rows.append(
        {
            "epoch": int(
                GRADIENT_AUDIT_EPOCH
            ),
            "batch": int(
                batch_id
            ),
            "batch_size": int(
                batch_features.shape[
                    0
                ]
            ),
            "writer_loss": float(
                writer_loss
                .detach()
                .item()
            ),
            "script_loss": float(
                script_loss
                .detach()
                .item()
            ),
            "writer_projection_gradient_norm": float(
                writer_gradient_norm
            ),
            "script_adversarial_projection_gradient_norm_at_grl1": float(
                adversarial_gradient_norm
            ),
            "raw_adversarial_to_writer_gradient_ratio": float(
                raw_gradient_ratio
            ),
            "writer_vs_reversed_script_gradient_cosine": float(
                writer_vs_adversarial_cosine
            ),
        }
    )


gradient_balance_df = pd.DataFrame(
    gradient_balance_rows
)


raw_ratio_values = (
    gradient_balance_df[
        "raw_adversarial_to_writer_gradient_ratio"
    ]
    .to_numpy()
)


RAW_GRADIENT_RATIO_MEDIAN = float(
    np.median(
        raw_ratio_values
    )
)


UNCLIPPED_GRL_STRENGTH = float(
    TARGET_ADVERSARIAL_TO_WRITER_GRAD_RATIO
    / RAW_GRADIENT_RATIO_MEDIAN
)


FROZEN_GRL_STRENGTH = float(
    np.clip(
        UNCLIPPED_GRL_STRENGTH,
        MIN_GRL_STRENGTH,
        MAX_GRL_STRENGTH,
    )
)


GRL_STRENGTH_CLIPPING_ACTIVATED = bool(
    not np.isclose(
        FROZEN_GRL_STRENGTH,
        UNCLIPPED_GRL_STRENGTH,
        rtol=0.0,
        atol=1e-15,
    )
)


gradient_balance_df[
    "scaled_adversarial_to_writer_gradient_ratio"
] = (
    gradient_balance_df[
        "raw_adversarial_to_writer_gradient_ratio"
    ]
    * FROZEN_GRL_STRENGTH
)


scaled_ratio_values = (
    gradient_balance_df[
        "scaled_adversarial_to_writer_gradient_ratio"
    ]
    .to_numpy()
)


scaled_ratio_median = float(
    np.median(
        scaled_ratio_values
    )
)

scaled_ratio_minimum = float(
    scaled_ratio_values.min()
)

scaled_ratio_maximum = float(
    scaled_ratio_values.max()
)


cosine_values = (
    gradient_balance_df[
        "writer_vs_reversed_script_gradient_cosine"
    ]
    .to_numpy()
)


gradient_balance_audit = {
    "purpose": (
        "freeze script-adversarial GRL strength "
        "using fit-only projection-gradient magnitudes"
    ),
    "gradient_audit_epoch": int(
        GRADIENT_AUDIT_EPOCH
    ),
    "gradient_audit_batches": [
        int(
            batch_id
        )
        for batch_id in (
            GRADIENT_AUDIT_BATCH_IDS
        )
    ],
    "gradient_audit_batch_count": int(
        len(
            gradient_balance_df
        )
    ),
    "script_head_initialization": (
        "fit-only competent warmed script head"
    ),
    "projection_initialization": (
        "same seed-42 Xavier initialization as Notebook 27"
    ),
    "target_adversarial_to_writer_gradient_ratio": float(
        TARGET_ADVERSARIAL_TO_WRITER_GRAD_RATIO
    ),
    "raw_gradient_ratio_mean": float(
        raw_ratio_values.mean()
    ),
    "raw_gradient_ratio_median": float(
        RAW_GRADIENT_RATIO_MEDIAN
    ),
    "raw_gradient_ratio_minimum": float(
        raw_ratio_values.min()
    ),
    "raw_gradient_ratio_maximum": float(
        raw_ratio_values.max()
    ),
    "unclipped_grl_strength": float(
        UNCLIPPED_GRL_STRENGTH
    ),
    "minimum_allowed_grl_strength": float(
        MIN_GRL_STRENGTH
    ),
    "maximum_allowed_grl_strength": float(
        MAX_GRL_STRENGTH
    ),
    "frozen_grl_strength": float(
        FROZEN_GRL_STRENGTH
    ),
    "grl_strength_clipping_activated": bool(
        GRL_STRENGTH_CLIPPING_ACTIVATED
    ),
    "scaled_gradient_ratio_mean": float(
        scaled_ratio_values.mean()
    ),
    "scaled_gradient_ratio_median": float(
        scaled_ratio_median
    ),
    "scaled_gradient_ratio_minimum": float(
        scaled_ratio_minimum
    ),
    "scaled_gradient_ratio_maximum": float(
        scaled_ratio_maximum
    ),
    "writer_vs_reversed_script_gradient_cosine_mean": float(
        cosine_values.mean()
    ),
    "writer_vs_reversed_script_gradient_cosine_median": float(
        np.median(
            cosine_values
        )
    ),
    "writer_vs_reversed_script_gradient_cosine_minimum": float(
        cosine_values.min()
    ),
    "writer_vs_reversed_script_gradient_cosine_maximum": float(
        cosine_values.max()
    ),
    "script_head_competence_verified": bool(
        script_head_competence_verified
    ),
    "grl_gradient_sign_verified": bool(
        grl_gradient_sign_verified
    ),
    "strength_selected_from_fit_only_gradients": True,
    "selection_used_to_choose_strength": False,
    "selection_evaluated": False,
    "monitor_used": False,
    "validation_used": False,
    "official_test_used": False,
    "optimizer_step_performed": False,
    "main_adversarial_training_started": False,
    "adversarial_strength_frozen": True,
}


gradient_balance_df.to_csv(
    REPORT_DIR
    / "fit_only_writer_vs_script_gradient_balance.csv",
    index=False,
)


with open(
    REPORT_DIR
    / "fit_only_gradient_balance_grl_strength.json",
    "w",
) as file:
    json.dump(
        gradient_balance_audit,
        file,
        indent=2,
    )


print(
    json.dumps(
        gradient_balance_audit,
        indent=2,
    )
)

print(
    "\nFit-only gradient balance:"
)

print(
    gradient_balance_df
    .round(
        6
    )
    .to_string(
        index=False
    )
)


if (
    len(
        gradient_balance_df
    ) != 6
    or not np.isfinite(
        gradient_balance_df[
            [
                "writer_projection_gradient_norm",
                "script_adversarial_projection_gradient_norm_at_grl1",
                "raw_adversarial_to_writer_gradient_ratio",
                "writer_vs_reversed_script_gradient_cosine",
                "scaled_adversarial_to_writer_gradient_ratio",
            ]
        ]
        .to_numpy()
    ).all()
    or (
        gradient_balance_df[
            "writer_projection_gradient_norm"
        ]
        <= 0.0
    ).any()
    or (
        gradient_balance_df[
            "script_adversarial_projection_gradient_norm_at_grl1"
        ]
        <= 0.0
    ).any()
    or FROZEN_GRL_STRENGTH
    < MIN_GRL_STRENGTH
    or FROZEN_GRL_STRENGTH
    > MAX_GRL_STRENGTH
    or not script_head_competence_verified
    or not grl_gradient_sign_verified
):
    raise RuntimeError(
        "Fit-only adversarial gradient-balance audit failed."
    )


if not GRL_STRENGTH_CLIPPING_ACTIVATED:
    if not np.isclose(
        scaled_ratio_median,
        TARGET_ADVERSARIAL_TO_WRITER_GRAD_RATIO,
        rtol=0.0,
        atol=1e-10,
    ):
        raise RuntimeError(
            "Frozen GRL strength did not reproduce the target median gradient ratio."
        )

{
  "purpose": "freeze script-adversarial GRL strength using fit-only projection-gradient magnitudes",
  "gradient_audit_epoch": 1,
  "gradient_audit_batches": [
    1,
    2,
    3,
    4,
    5,
    6
  ],
  "gradient_audit_batch_count": 6,
  "script_head_initialization": "fit-only competent warmed script head",
  "projection_initialization": "same seed-42 Xavier initialization as Notebook 27",
  "target_adversarial_to_writer_gradient_ratio": 0.2,
  "raw_gradient_ratio_mean": 2.1341671322849893,
  "raw_gradient_ratio_median": 1.7674294341790118,
  "raw_gradient_ratio_minimum": 1.2840460530709301,
  "raw_gradient_ratio_maximum": 3.4703310866983035,
  "unclipped_grl_strength": 0.11315869031733194,
  "minimum_allowed_grl_strength": 0.05,
  "maximum_allowed_grl_strength": 4.0,
  "frozen_grl_strength": 0.11315869031733194,
  "grl_strength_clipping_activated": false,
  "scaled_gradient_ratio_mean": 0.2414995576076655,
  "scaled_gradient_ratio_median": 0.2,
  "scaled_gradient_ratio_minimum"

In [9]:
ADVERSARIAL_PROJECTION_LR = float(
    baseline_checkpoint[
        "learning_rate"
    ]
)

ADVERSARIAL_SCRIPT_HEAD_LR = float(
    SCRIPT_HEAD_LR
)

ADVERSARIAL_WEIGHT_DECAY = float(
    baseline_checkpoint[
        "weight_decay"
    ]
)

ADVERSARIAL_GRAD_CLIP_MAX_NORM = 5.0


def build_main_adversarial_model(
    seed,
):
    set_experiment_seed(
        seed
    )

    model = (
        DINOv2SScriptAdversarialStudent(
            input_dimension=(
                DINO_S_FEATURE_DIM
            ),
            projection_dimension=(
                PROJECTION_DIM
            ),
            script_classes=(
                SCRIPT_CLASSES
            ),
        )
    )

    model.script_head.load_state_dict(
        script_head_warmup_model
        .script_head
        .state_dict(),
        strict=True,
    )

    return model


def build_main_adversarial_optimizers(
    model,
):
    projection_optimizer = (
        torch.optim.AdamW(
            model.projection.parameters(),
            lr=(
                ADVERSARIAL_PROJECTION_LR
            ),
            weight_decay=(
                ADVERSARIAL_WEIGHT_DECAY
            ),
        )
    )

    script_head_optimizer = (
        torch.optim.AdamW(
            model.script_head.parameters(),
            lr=(
                ADVERSARIAL_SCRIPT_HEAD_LR
            ),
            weight_decay=(
                ADVERSARIAL_WEIGHT_DECAY
            ),
        )
    )

    return (
        projection_optimizer,
        script_head_optimizer,
    )


def parameter_state(
    module,
):
    return {
        name: parameter
        .detach()
        .clone()
        for name, parameter in (
            module.named_parameters()
        )
    }


def parameter_change_summary(
    module,
    reference_state,
):
    squared_sum = 0.0
    maximum_absolute_update = 0.0
    changed_tensors = 0

    rows = []

    for name, parameter in (
        module.named_parameters()
    ):
        difference = (
            parameter
            .detach()
            - reference_state[
                name
            ]
        )

        update_norm = float(
            difference
            .double()
            .norm()
            .item()
        )

        maximum_update = float(
            difference
            .abs()
            .max()
            .item()
        )

        changed = bool(
            maximum_update
            > 0.0
        )

        squared_sum += float(
            difference
            .double()
            .pow(
                2
            )
            .sum()
            .item()
        )

        maximum_absolute_update = max(
            maximum_absolute_update,
            maximum_update,
        )

        changed_tensors += int(
            changed
        )

        rows.append(
            {
                "parameter": name,
                "update_norm": float(
                    update_norm
                ),
                "maximum_absolute_update": float(
                    maximum_update
                ),
                "changed": bool(
                    changed
                ),
            }
        )

    return {
        "total_update_norm": float(
            np.sqrt(
                squared_sum
            )
        ),
        "maximum_absolute_update": float(
            maximum_absolute_update
        ),
        "changed_tensors": int(
            changed_tensors
        ),
        "rows": rows,
    }


alternating_sanity_model = (
    build_main_adversarial_model(
        ADVERSARIAL_SEED
    )
)

(
    alternating_projection_optimizer,
    alternating_script_optimizer,
) = (
    build_main_adversarial_optimizers(
        alternating_sanity_model
    )
)


sanity_batch = (
    load_fit_writer_batch(
        epoch=1,
        batch_id=1,
    )
)

sanity_batch_features = (
    torch.from_numpy(
        sanity_batch[
            "features"
        ]
    )
    .float()
)

sanity_batch_labels = (
    sanity_batch[
        "script_labels"
    ]
)


initial_projection_state = (
    parameter_state(
        alternating_sanity_model
        .projection
    )
)

initial_script_head_state = (
    parameter_state(
        alternating_sanity_model
        .script_head
    )
)


alternating_sanity_model.eval()

with torch.no_grad():
    initial_embeddings = (
        alternating_sanity_model
        .project(
            sanity_batch_features
        )
    )

    initial_writer_objective = (
        build_writer_objective(
            initial_embeddings
        )
    )

    initial_script_logits = (
        alternating_sanity_model
        .classify_script(
            initial_embeddings,
            use_grl=False,
        )
    )

    initial_script_loss = float(
        F.cross_entropy(
            initial_script_logits.reshape(
                -1,
                SCRIPT_CLASSES,
            ),
            sanity_batch_labels.reshape(
                -1
            ),
        )
        .item()
    )


for parameter in (
    alternating_sanity_model
    .projection
    .parameters()
):
    parameter.requires_grad = False

for parameter in (
    alternating_sanity_model
    .script_head
    .parameters()
):
    parameter.requires_grad = True


alternating_sanity_model.train()

alternating_script_optimizer.zero_grad(
    set_to_none=True
)

with torch.no_grad():
    head_step_embeddings = (
        alternating_sanity_model
        .project(
            sanity_batch_features
        )
    )

head_step_logits = (
    alternating_sanity_model
    .classify_script(
        head_step_embeddings,
        use_grl=False,
    )
)

head_step_script_loss = (
    F.cross_entropy(
        head_step_logits.reshape(
            -1,
            SCRIPT_CLASSES,
        ),
        sanity_batch_labels.reshape(
            -1
        ),
    )
)

head_step_script_loss.backward()

head_gradient_norm_squared = 0.0

for parameter in (
    alternating_sanity_model
    .script_head
    .parameters()
):
    if parameter.grad is None:
        raise RuntimeError(
            "Missing script-head gradient."
        )

    head_gradient_norm_squared += float(
        parameter.grad
        .detach()
        .double()
        .pow(
            2
        )
        .sum()
        .item()
    )

head_gradient_norm = float(
    np.sqrt(
        head_gradient_norm_squared
    )
)

alternating_script_optimizer.step()


projection_change_after_head_step = (
    parameter_change_summary(
        alternating_sanity_model
        .projection,
        initial_projection_state,
    )
)

script_change_after_head_step = (
    parameter_change_summary(
        alternating_sanity_model
        .script_head,
        initial_script_head_state,
    )
)


script_head_state_after_head_step = (
    parameter_state(
        alternating_sanity_model
        .script_head
    )
)


alternating_sanity_model.eval()

with torch.no_grad():
    post_head_embeddings = (
        alternating_sanity_model
        .project(
            sanity_batch_features
        )
    )

    post_head_logits = (
        alternating_sanity_model
        .classify_script(
            post_head_embeddings,
            use_grl=False,
        )
    )

    post_head_script_loss = float(
        F.cross_entropy(
            post_head_logits.reshape(
                -1,
                SCRIPT_CLASSES,
            ),
            sanity_batch_labels.reshape(
                -1
            ),
        )
        .item()
    )


for parameter in (
    alternating_sanity_model
    .projection
    .parameters()
):
    parameter.requires_grad = True

for parameter in (
    alternating_sanity_model
    .script_head
    .parameters()
):
    parameter.requires_grad = False


alternating_sanity_model.train()

alternating_projection_optimizer.zero_grad(
    set_to_none=True
)

projection_step_output = (
    alternating_sanity_model(
        sanity_batch_features,
        grl_strength=(
            FROZEN_GRL_STRENGTH
        ),
        use_grl=True,
    )
)

projection_step_writer_objective = (
    build_writer_objective(
        projection_step_output[
            "embeddings"
        ]
    )
)

projection_step_writer_loss = (
    projection_step_writer_objective[
        "base_loss"
    ]
)

projection_step_script_loss = (
    F.cross_entropy(
        projection_step_output[
            "script_logits"
        ].reshape(
            -1,
            SCRIPT_CLASSES,
        ),
        sanity_batch_labels.reshape(
            -1
        ),
    )
)

projection_step_total_loss = (
    projection_step_writer_loss
    + projection_step_script_loss
)

projection_step_total_loss.backward()


projection_gradient_norm_before_clip = (
    torch.nn.utils.clip_grad_norm_(
        alternating_sanity_model
        .projection
        .parameters(),
        max_norm=(
            ADVERSARIAL_GRAD_CLIP_MAX_NORM
        ),
    )
)

projection_gradient_norm_before_clip = float(
    projection_gradient_norm_before_clip
    .detach()
    .item()
)


alternating_projection_optimizer.step()


projection_change_after_projection_step = (
    parameter_change_summary(
        alternating_sanity_model
        .projection,
        initial_projection_state,
    )
)

script_change_during_projection_step = (
    parameter_change_summary(
        alternating_sanity_model
        .script_head,
        script_head_state_after_head_step,
    )
)


alternating_sanity_model.eval()

with torch.no_grad():
    final_embeddings = (
        alternating_sanity_model
        .project(
            sanity_batch_features
        )
    )

    final_writer_objective = (
        build_writer_objective(
            final_embeddings
        )
    )

    final_script_logits = (
        alternating_sanity_model
        .classify_script(
            final_embeddings,
            use_grl=False,
        )
    )

    final_script_loss = float(
        F.cross_entropy(
            final_script_logits.reshape(
                -1,
                SCRIPT_CLASSES,
            ),
            sanity_batch_labels.reshape(
                -1
            ),
        )
        .item()
    )


final_embedding_norms = (
    torch.linalg.vector_norm(
        final_embeddings,
        dim=-1,
    )
)


alternating_parameter_rows = []

for row in (
    projection_change_after_head_step[
        "rows"
    ]
):
    alternating_parameter_rows.append(
        {
            "stage": (
                "after_script_head_step"
            ),
            "parameter_group": (
                "projection"
            ),
            **row,
        }
    )

for row in (
    script_change_after_head_step[
        "rows"
    ]
):
    alternating_parameter_rows.append(
        {
            "stage": (
                "after_script_head_step"
            ),
            "parameter_group": (
                "script_head"
            ),
            **row,
        }
    )

for row in (
    projection_change_after_projection_step[
        "rows"
    ]
):
    alternating_parameter_rows.append(
        {
            "stage": (
                "after_projection_step"
            ),
            "parameter_group": (
                "projection"
            ),
            **row,
        }
    )

for row in (
    script_change_during_projection_step[
        "rows"
    ]
):
    alternating_parameter_rows.append(
        {
            "stage": (
                "during_projection_step"
            ),
            "parameter_group": (
                "script_head"
            ),
            **row,
        }
    )


alternating_optimizer_parameter_df = (
    pd.DataFrame(
        alternating_parameter_rows
    )
)


alternating_optimizer_sanity = {
    "method": (
        "alternating DINOv2-S script-adversarial projection"
    ),
    "batch_writers": (
        sanity_batch[
            "writers"
        ]
    ),
    "batch_size": int(
        sanity_batch_features.shape[
            0
        ]
    ),
    "projection_learning_rate": float(
        ADVERSARIAL_PROJECTION_LR
    ),
    "script_head_learning_rate": float(
        ADVERSARIAL_SCRIPT_HEAD_LR
    ),
    "weight_decay": float(
        ADVERSARIAL_WEIGHT_DECAY
    ),
    "frozen_grl_strength": float(
        FROZEN_GRL_STRENGTH
    ),
    "initial_writer_loss": float(
        initial_writer_objective[
            "base_loss"
        ].item()
    ),
    "expected_notebook27_initial_writer_loss": float(
        expected_notebook27_initial_base_loss
    ),
    "initial_script_loss": float(
        initial_script_loss
    ),
    "script_head_step_loss": float(
        head_step_script_loss
        .detach()
        .item()
    ),
    "script_head_gradient_norm": float(
        head_gradient_norm
    ),
    "post_head_step_script_loss_same_batch": float(
        post_head_script_loss
    ),
    "script_loss_change_after_head_step": float(
        post_head_script_loss
        - initial_script_loss
    ),
    "projection_update_norm_during_head_step": float(
        projection_change_after_head_step[
            "total_update_norm"
        ]
    ),
    "script_head_update_norm_during_head_step": float(
        script_change_after_head_step[
            "total_update_norm"
        ]
    ),
    "projection_step_writer_loss": float(
        projection_step_writer_loss
        .detach()
        .item()
    ),
    "projection_step_script_loss": float(
        projection_step_script_loss
        .detach()
        .item()
    ),
    "projection_step_total_loss": float(
        projection_step_total_loss
        .detach()
        .item()
    ),
    "projection_gradient_norm_before_clip": float(
        projection_gradient_norm_before_clip
    ),
    "gradient_clip_threshold": float(
        ADVERSARIAL_GRAD_CLIP_MAX_NORM
    ),
    "gradient_clipping_activated": bool(
        projection_gradient_norm_before_clip
        > ADVERSARIAL_GRAD_CLIP_MAX_NORM
    ),
    "projection_update_norm_during_projection_step": float(
        projection_change_after_projection_step[
            "total_update_norm"
        ]
    ),
    "script_head_update_norm_during_projection_step": float(
        script_change_during_projection_step[
            "total_update_norm"
        ]
    ),
    "final_writer_loss_same_batch": float(
        final_writer_objective[
            "base_loss"
        ].item()
    ),
    "final_script_loss_same_batch": float(
        final_script_loss
    ),
    "final_embedding_norm_min": float(
        final_embedding_norms
        .min()
        .item()
    ),
    "final_embedding_norm_max": float(
        final_embedding_norms
        .max()
        .item()
    ),
    "head_step_updates_only_script_head": bool(
        projection_change_after_head_step[
            "total_update_norm"
        ]
        == 0.0
        and script_change_after_head_step[
            "total_update_norm"
        ]
        > 0.0
    ),
    "projection_step_updates_only_projection": bool(
        projection_change_after_projection_step[
            "total_update_norm"
        ]
        > 0.0
        and script_change_during_projection_step[
            "total_update_norm"
        ]
        == 0.0
    ),
    "alternating_optimizer_plumbing_verified": True,
    "optimizer_steps_per_training_batch": 2,
    "script_head_steps_per_training_batch": 1,
    "projection_steps_per_training_batch": 1,
    "full_epoch_executed": False,
    "main_adversarial_training_started": False,
    "selection_used": False,
    "selection_evaluated": False,
    "monitor_used": False,
    "validation_used": False,
    "official_test_used": False,
}


alternating_optimizer_parameter_df.to_csv(
    REPORT_DIR
    / "alternating_optimizer_one_batch_parameter_audit.csv",
    index=False,
)


with open(
    REPORT_DIR
    / "alternating_optimizer_one_batch_sanity.json",
    "w",
) as file:
    json.dump(
        alternating_optimizer_sanity,
        file,
        indent=2,
    )


print(
    json.dumps(
        alternating_optimizer_sanity,
        indent=2,
    )
)

print(
    "\nAlternating optimizer parameter audit:"
)

print(
    alternating_optimizer_parameter_df
    .round(
        9
    )
    .to_string(
        index=False
    )
)


if (
    abs(
        alternating_optimizer_sanity[
            "initial_writer_loss"
        ]
        - expected_notebook27_initial_base_loss
    )
    > 1e-7
    or alternating_optimizer_sanity[
        "projection_update_norm_during_head_step"
    ]
    != 0.0
    or alternating_optimizer_sanity[
        "script_head_update_norm_during_head_step"
    ]
    <= 0.0
    or alternating_optimizer_sanity[
        "projection_update_norm_during_projection_step"
    ]
    <= 0.0
    or alternating_optimizer_sanity[
        "script_head_update_norm_during_projection_step"
    ]
    != 0.0
    or not np.isfinite(
        [
            alternating_optimizer_sanity[
                "script_head_gradient_norm"
            ],
            alternating_optimizer_sanity[
                "projection_gradient_norm_before_clip"
            ],
            alternating_optimizer_sanity[
                "final_writer_loss_same_batch"
            ],
            alternating_optimizer_sanity[
                "final_script_loss_same_batch"
            ],
        ]
    ).all()
    or not torch.allclose(
        final_embedding_norms,
        torch.ones_like(
            final_embedding_norms
        ),
        rtol=0.0,
        atol=1e-5,
    )
):
    raise RuntimeError(
        "Alternating optimizer one-batch sanity failed."
    )

{
  "method": "alternating DINOv2-S script-adversarial projection",
  "batch_writers": [
    133,
    39,
    273,
    138,
    172,
    249,
    272,
    94
  ],
  "batch_size": 8,
  "projection_learning_rate": 0.001,
  "script_head_learning_rate": 0.001,
  "weight_decay": 0.0001,
  "frozen_grl_strength": 0.11315869031733194,
  "initial_writer_loss": 0.45000821352005005,
  "expected_notebook27_initial_writer_loss": 0.45000821352005005,
  "initial_script_loss": 0.5931294560432434,
  "script_head_step_loss": 0.5931294560432434,
  "script_head_gradient_norm": 0.054219490990998424,
  "post_head_step_script_loss_same_batch": 0.5924104452133179,
  "script_loss_change_after_head_step": -0.0007190108299255371,
  "projection_update_norm_during_head_step": 0.0,
  "script_head_update_norm_during_head_step": 0.01702881306945254,
  "projection_step_writer_loss": 0.45000821352005005,
  "projection_step_script_loss": 0.5924104452133179,
  "projection_step_total_loss": 1.0424187183380127,
  "projecti

In [10]:
ADVERSARIAL_EPOCHS = 10

SCRIPT_AUC_MINIMUM_REDUCTION = 0.05
WRITER_MACRO_AUC_MAXIMUM_DROP = 0.005
WRITER_CONDITION_AUC_MAXIMUM_DROP = 0.02

SCRIPT_AUC_SUCCESS_THRESHOLD = float(
    BASELINE_SCRIPT_PROBE_AUC
    - SCRIPT_AUC_MINIMUM_REDUCTION
)

WRITER_MACRO_AUC_PRESERVATION_THRESHOLD = float(
    BASELINE_SEED42_MACRO_AUC
    - WRITER_MACRO_AUC_MAXIMUM_DROP
)

baseline_condition_auc_map = {
    str(
        row[
            "condition"
        ]
    ): float(
        row[
            "auc"
        ]
    )
    for _, row in (
        baseline_condition_df
        .iterrows()
    )
}


def train_alternating_adversarial_batch(
    model,
    projection_optimizer,
    script_optimizer,
    epoch,
    batch_id,
):
    batch = (
        load_fit_writer_batch(
            epoch=epoch,
            batch_id=batch_id,
        )
    )

    features = (
        torch.from_numpy(
            batch[
                "features"
            ]
        )
        .float()
    )

    script_labels = (
        batch[
            "script_labels"
        ]
    )

    for parameter in (
        model.projection.parameters()
    ):
        parameter.requires_grad = False

    for parameter in (
        model.script_head.parameters()
    ):
        parameter.requires_grad = True

    model.train()

    script_optimizer.zero_grad(
        set_to_none=True
    )

    with torch.no_grad():
        head_embeddings = (
            model.project(
                features
            )
        )

    head_logits = (
        model.classify_script(
            head_embeddings,
            use_grl=False,
        )
    )

    head_loss = (
        F.cross_entropy(
            head_logits.reshape(
                -1,
                SCRIPT_CLASSES,
            ),
            script_labels.reshape(
                -1
            ),
        )
    )

    head_loss.backward()

    script_head_gradient_norm_squared = 0.0

    for parameter in (
        model.script_head.parameters()
    ):
        if parameter.grad is None:
            raise RuntimeError(
                "Missing script-head gradient during alternating training."
            )

        script_head_gradient_norm_squared += float(
            parameter.grad
            .detach()
            .double()
            .pow(
                2
            )
            .sum()
            .item()
        )

    script_head_gradient_norm = float(
        np.sqrt(
            script_head_gradient_norm_squared
        )
    )

    script_optimizer.step()

    head_predictions = (
        head_logits
        .argmax(
            dim=-1
        )
    )

    head_correct = int(
        (
            head_predictions
            == script_labels
        )
        .sum()
        .item()
    )

    head_examples = int(
        script_labels.numel()
    )


    for parameter in (
        model.projection.parameters()
    ):
        parameter.requires_grad = True

    for parameter in (
        model.script_head.parameters()
    ):
        parameter.requires_grad = False

    projection_optimizer.zero_grad(
        set_to_none=True
    )

    projection_output = (
        model(
            features,
            grl_strength=(
                FROZEN_GRL_STRENGTH
            ),
            use_grl=True,
        )
    )

    writer_objective = (
        build_writer_objective(
            projection_output[
                "embeddings"
            ]
        )
    )

    writer_loss = (
        writer_objective[
            "base_loss"
        ]
    )

    adversarial_script_loss = (
        F.cross_entropy(
            projection_output[
                "script_logits"
            ].reshape(
                -1,
                SCRIPT_CLASSES,
            ),
            script_labels.reshape(
                -1
            ),
        )
    )

    projection_total_loss = (
        writer_loss
        + adversarial_script_loss
    )

    projection_total_loss.backward()

    projection_gradient_norm = (
        torch.nn.utils.clip_grad_norm_(
            model.projection.parameters(),
            max_norm=(
                ADVERSARIAL_GRAD_CLIP_MAX_NORM
            ),
        )
    )

    projection_gradient_norm = float(
        projection_gradient_norm
        .detach()
        .item()
    )

    projection_optimizer.step()


    for parameter in (
        model.script_head.parameters()
    ):
        parameter.requires_grad = True

    model.eval()

    with torch.no_grad():
        post_step_embeddings = (
            model.project(
                features
            )
        )

        post_step_script_logits = (
            model.classify_script(
                post_step_embeddings,
                use_grl=False,
            )
        )

        post_step_script_loss = float(
            F.cross_entropy(
                post_step_script_logits.reshape(
                    -1,
                    SCRIPT_CLASSES,
                ),
                script_labels.reshape(
                    -1
                ),
            )
            .item()
        )

        post_step_script_predictions = (
            post_step_script_logits
            .argmax(
                dim=-1
            )
        )

        post_step_script_accuracy = float(
            (
                post_step_script_predictions
                == script_labels
            )
            .float()
            .mean()
            .item()
        )


    return {
        "epoch": int(
            epoch
        ),
        "batch": int(
            batch_id
        ),
        "batch_size": int(
            features.shape[
                0
            ]
        ),
        "writers": (
            batch[
                "writers"
            ]
        ),
        "writer_base_loss": float(
            writer_loss
            .detach()
            .item()
        ),
        "arabic_metric_loss": float(
            writer_objective[
                "arabic_result"
            ][
                "loss"
            ]
            .detach()
            .item()
        ),
        "cross_metric_loss": float(
            writer_objective[
                "cross_metric_loss"
            ]
            .detach()
            .item()
        ),
        "script_head_step_loss": float(
            head_loss
            .detach()
            .item()
        ),
        "script_head_step_accuracy": float(
            head_correct
            / head_examples
        ),
        "script_head_gradient_norm": float(
            script_head_gradient_norm
        ),
        "projection_step_script_loss": float(
            adversarial_script_loss
            .detach()
            .item()
        ),
        "projection_step_total_loss": float(
            projection_total_loss
            .detach()
            .item()
        ),
        "projection_gradient_norm_before_clip": float(
            projection_gradient_norm
        ),
        "projection_gradient_clipping_activated": bool(
            projection_gradient_norm
            > ADVERSARIAL_GRAD_CLIP_MAX_NORM
        ),
        "post_step_script_loss": float(
            post_step_script_loss
        ),
        "post_step_script_accuracy": float(
            post_step_script_accuracy
        ),
    }


def train_alternating_adversarial_epoch(
    model,
    projection_optimizer,
    script_optimizer,
    epoch,
):
    epoch_batch_ids = (
        fit_schedule_df[
            fit_schedule_df[
                "epoch"
            ] == epoch
        ][
            "batch"
        ]
        .drop_duplicates()
        .sort_values()
        .astype(int)
        .tolist()
    )

    batch_rows = []

    for batch_id in (
        epoch_batch_ids
    ):
        batch_result = (
            train_alternating_adversarial_batch(
                model=model,
                projection_optimizer=(
                    projection_optimizer
                ),
                script_optimizer=(
                    script_optimizer
                ),
                epoch=epoch,
                batch_id=batch_id,
            )
        )

        batch_rows.append(
            batch_result
        )

    batch_df = pd.DataFrame(
        [
            {
                key: value
                for key, value in (
                    row.items()
                )
                if key != "writers"
            }
            for row in (
                batch_rows
            )
        ]
    )

    epoch_summary = {
        "epoch": int(
            epoch
        ),
        "batches": int(
            len(
                batch_df
            )
        ),
        "writers_seen": int(
            batch_df[
                "batch_size"
            ].sum()
        ),
        "writer_base_loss": float(
            batch_df[
                "writer_base_loss"
            ].mean()
        ),
        "arabic_metric_loss": float(
            batch_df[
                "arabic_metric_loss"
            ].mean()
        ),
        "cross_metric_loss": float(
            batch_df[
                "cross_metric_loss"
            ].mean()
        ),
        "script_head_step_loss": float(
            batch_df[
                "script_head_step_loss"
            ].mean()
        ),
        "script_head_step_accuracy": float(
            np.average(
                batch_df[
                    "script_head_step_accuracy"
                ],
                weights=(
                    batch_df[
                        "batch_size"
                    ]
                ),
            )
        ),
        "post_step_script_loss": float(
            batch_df[
                "post_step_script_loss"
            ].mean()
        ),
        "post_step_script_accuracy": float(
            np.average(
                batch_df[
                    "post_step_script_accuracy"
                ],
                weights=(
                    batch_df[
                        "batch_size"
                    ]
                ),
            )
        ),
        "script_head_gradient_norm_mean": float(
            batch_df[
                "script_head_gradient_norm"
            ].mean()
        ),
        "projection_gradient_norm_mean": float(
            batch_df[
                "projection_gradient_norm_before_clip"
            ].mean()
        ),
        "projection_gradient_norm_max": float(
            batch_df[
                "projection_gradient_norm_before_clip"
            ].max()
        ),
        "projection_gradient_clipping_events": int(
            batch_df[
                "projection_gradient_clipping_activated"
            ].sum()
        ),
    }

    return (
        epoch_summary,
        batch_df,
    )


def build_fixed_adversarial_experiment(
    seed,
):
    model = (
        build_main_adversarial_model(
            seed
        )
    )

    (
        projection_optimizer,
        script_optimizer,
    ) = (
        build_main_adversarial_optimizers(
            model
        )
    )

    return (
        model,
        projection_optimizer,
        script_optimizer,
    )


adversarial_protocol_configuration = {
    "method": (
        "DINOv2-S script-adversarial projection"
    ),
    "development_seed": int(
        ADVERSARIAL_SEED
    ),
    "input_dimension": int(
        DINO_S_FEATURE_DIM
    ),
    "projection_dimension": int(
        PROJECTION_DIM
    ),
    "projection_parameters": 55440,
    "script_head_architecture": (
        "Linear(144, 2)"
    ),
    "script_head_parameters": 290,
    "total_trainable_parameters": 55730,
    "projection_initialization": (
        "same seed-specific Xavier initialization "
        "as Notebook 27 projection baseline"
    ),
    "script_head_initialization": (
        "fit-only competent warmed head"
    ),
    "script_head_warmup_epochs": int(
        SCRIPT_HEAD_WARMUP_EPOCHS
    ),
    "writer_training_epochs": int(
        ADVERSARIAL_EPOCHS
    ),
    "fit_writers": 145,
    "selection_writers": 36,
    "batches_per_epoch": 18,
    "projection_learning_rate": float(
        ADVERSARIAL_PROJECTION_LR
    ),
    "script_head_learning_rate": float(
        ADVERSARIAL_SCRIPT_HEAD_LR
    ),
    "weight_decay": float(
        ADVERSARIAL_WEIGHT_DECAY
    ),
    "cosine_margin": float(
        COSINE_MARGIN
    ),
    "gradient_clip_max_norm": float(
        ADVERSARIAL_GRAD_CLIP_MAX_NORM
    ),
    "frozen_grl_strength": float(
        FROZEN_GRL_STRENGTH
    ),
    "grl_strength_source": (
        "fit-only six-batch median projection-gradient balance"
    ),
    "target_adversarial_to_writer_gradient_ratio": float(
        TARGET_ADVERSARIAL_TO_WRITER_GRAD_RATIO
    ),
    "training_schedule": (
        "one script-head optimizer step followed by "
        "one writer-plus-GRL projection optimizer step per batch"
    ),
    "writer_objective_matches_notebook27": True,
    "script_is_only_adversarial_target": True,
    "text_condition_is_diagnostic_only": True,
    "fixed_epoch10_evaluation": True,
    "selection_checkpointing_used": False,
    "early_stopping_used": False,
    "epoch_extension_allowed": False,
    "baseline_seed42_writer_macro_auc": float(
        BASELINE_SEED42_MACRO_AUC
    ),
    "baseline_script_probe_auc": float(
        BASELINE_SCRIPT_PROBE_AUC
    ),
    "minimum_script_auc_reduction_for_pareto_support": float(
        SCRIPT_AUC_MINIMUM_REDUCTION
    ),
    "script_auc_success_threshold": float(
        SCRIPT_AUC_SUCCESS_THRESHOLD
    ),
    "maximum_writer_macro_auc_drop_for_pareto_support": float(
        WRITER_MACRO_AUC_MAXIMUM_DROP
    ),
    "writer_macro_auc_preservation_threshold": float(
        WRITER_MACRO_AUC_PRESERVATION_THRESHOLD
    ),
    "maximum_individual_condition_auc_drop_for_pareto_support": float(
        WRITER_CONDITION_AUC_MAXIMUM_DROP
    ),
    "pareto_support_rule": (
        "script probe AUC reduction >= 0.05 AND "
        "writer macro AUC drop <= 0.005 AND "
        "no cross-script condition AUC drop > 0.02"
    ),
    "tradeoff_interpretation_rule": (
        "if script accessibility decreases materially but "
        "writer preservation guardrails fail, classify as "
        "an invariance-performance tradeoff rather than "
        "a baseline replacement"
    ),
    "attached_head_confusion_will_not_be_used_as_invariance_evidence": True,
    "independent_posthoc_probe_required": True,
    "script_head_competence_verified": bool(
        script_head_competence_verified
    ),
    "grl_gradient_sign_verified": bool(
        grl_gradient_sign_verified
    ),
    "alternating_optimizer_plumbing_verified": bool(
        alternating_optimizer_sanity[
            "alternating_optimizer_plumbing_verified"
        ]
    ),
    "adversarial_strength_frozen": True,
    "architecture_frozen": True,
    "optimizer_protocol_frozen": True,
    "decision_thresholds_frozen": True,
    "main_training_executed_in_this_cell": False,
    "selection_evaluated_in_this_cell": False,
    "monitor_used": False,
    "validation_used": False,
    "official_test_used": False,
}

with open(
    REPORT_DIR
    / "dinov2s_script_adversarial_protocol_configuration.json",
    "w",
) as file:
    json.dump(
        adversarial_protocol_configuration,
        file,
        indent=2,
    )

print(
    json.dumps(
        adversarial_protocol_configuration,
        indent=2,
    )
)

if (
    ADVERSARIAL_EPOCHS
    != 10
    or not np.isclose(
        ADVERSARIAL_PROJECTION_LR,
        0.001,
        rtol=0.0,
        atol=1e-15,
    )
    or not np.isclose(
        ADVERSARIAL_SCRIPT_HEAD_LR,
        0.001,
        rtol=0.0,
        atol=1e-15,
    )
    or not script_head_competence_verified
    or not grl_gradient_sign_verified
    or not alternating_optimizer_sanity[
        "alternating_optimizer_plumbing_verified"
    ]
    or not adversarial_protocol_configuration[
        "adversarial_strength_frozen"
    ]
    or not adversarial_protocol_configuration[
        "decision_thresholds_frozen"
    ]
):
    raise RuntimeError(
        "Notebook 29 adversarial protocol freeze failed."
    )

{
  "method": "DINOv2-S script-adversarial projection",
  "development_seed": 42,
  "input_dimension": 384,
  "projection_dimension": 144,
  "projection_parameters": 55440,
  "script_head_architecture": "Linear(144, 2)",
  "script_head_parameters": 290,
  "total_trainable_parameters": 55730,
  "projection_initialization": "same seed-specific Xavier initialization as Notebook 27 projection baseline",
  "script_head_initialization": "fit-only competent warmed head",
  "script_head_warmup_epochs": 10,
  "writer_training_epochs": 10,
  "fit_writers": 145,
  "selection_writers": 36,
  "batches_per_epoch": 18,
  "projection_learning_rate": 0.001,
  "script_head_learning_rate": 0.001,
  "weight_decay": 0.0001,
  "cosine_margin": 0.5,
  "gradient_clip_max_norm": 5.0,
  "frozen_grl_strength": 0.11315869031733194,
  "grl_strength_source": "fit-only six-batch median projection-gradient balance",
  "target_adversarial_to_writer_gradient_ratio": 0.2,
  "training_schedule": "one script-head optimize

In [11]:
MAIN_ADVERSARIAL_SEED = 42

main_adversarial_model, (
    main_projection_optimizer
), (
    main_script_optimizer
) = (
    build_fixed_adversarial_experiment(
        MAIN_ADVERSARIAL_SEED
    )
)


main_adversarial_model.eval()

with torch.no_grad():
    pretraining_first_batch_embeddings = (
        main_adversarial_model
        .project(
            first_batch_tensor
        )
    )

    pretraining_first_batch_writer_objective = (
        build_writer_objective(
            pretraining_first_batch_embeddings
        )
    )

    pretraining_first_batch_script_logits = (
        main_adversarial_model
        .classify_script(
            pretraining_first_batch_embeddings,
            use_grl=False,
        )
    )

    pretraining_first_batch_script_loss = float(
        F.cross_entropy(
            pretraining_first_batch_script_logits.reshape(
                -1,
                SCRIPT_CLASSES,
            ),
            first_batch_script_labels.reshape(
                -1
            ),
        )
        .item()
    )


pretraining_writer_loss = float(
    pretraining_first_batch_writer_objective[
        "base_loss"
    ]
    .item()
)

pretraining_writer_loss_difference = float(
    pretraining_writer_loss
    - expected_notebook27_initial_base_loss
)


main_adversarial_history_rows = []

main_adversarial_batch_rows = []


for epoch in range(
    1,
    ADVERSARIAL_EPOCHS
    + 1,
):
    epoch_summary, epoch_batch_df = (
        train_alternating_adversarial_epoch(
            model=(
                main_adversarial_model
            ),
            projection_optimizer=(
                main_projection_optimizer
            ),
            script_optimizer=(
                main_script_optimizer
            ),
            epoch=epoch,
        )
    )

    if (
        epoch_summary[
            "batches"
        ] != 18
        or epoch_summary[
            "writers_seen"
        ] != 145
    ):
        raise RuntimeError(
            "Adversarial epoch schedule audit failed."
        )

    main_adversarial_history_rows.append(
        epoch_summary
    )

    epoch_batch_df = (
        epoch_batch_df
        .copy()
    )

    epoch_batch_df[
        "training_seed"
    ] = int(
        MAIN_ADVERSARIAL_SEED
    )

    main_adversarial_batch_rows.append(
        epoch_batch_df
    )

    print(
        f"dinov2s-script-adv | "
        f"seed {MAIN_ADVERSARIAL_SEED} | "
        f"epoch {epoch:02d} | "
        f"writer "
        f"{epoch_summary['writer_base_loss']:.6f} | "
        f"arabic "
        f"{epoch_summary['arabic_metric_loss']:.6f} | "
        f"cross "
        f"{epoch_summary['cross_metric_loss']:.6f} | "
        f"head loss "
        f"{epoch_summary['script_head_step_loss']:.6f} | "
        f"head acc "
        f"{epoch_summary['script_head_step_accuracy']:.4f} | "
        f"post acc "
        f"{epoch_summary['post_step_script_accuracy']:.4f} | "
        f"proj grad "
        f"{epoch_summary['projection_gradient_norm_mean']:.6f}"
    )


main_adversarial_history_df = (
    pd.DataFrame(
        main_adversarial_history_rows
    )
)

main_adversarial_batch_df = (
    pd.concat(
        main_adversarial_batch_rows,
        ignore_index=True,
    )
)


ADVERSARIAL_CHECKPOINT_PATH = (
    CHECKPOINT_DIR
    / "dinov2s_script_adversarial_seed42_epoch10.pt"
)

ADVERSARIAL_HISTORY_PATH = (
    REPORT_DIR
    / "dinov2s_script_adversarial_seed42_history.csv"
)

ADVERSARIAL_BATCH_HISTORY_PATH = (
    REPORT_DIR
    / "dinov2s_script_adversarial_seed42_batches.csv"
)


torch.save(
    {
        "method": (
            "DINOv2-S script-adversarial projection"
        ),
        "training_seed": int(
            MAIN_ADVERSARIAL_SEED
        ),
        "epoch": int(
            ADVERSARIAL_EPOCHS
        ),
        "epochs_completed": int(
            ADVERSARIAL_EPOCHS
        ),
        "model_state_dict": (
            main_adversarial_model
            .state_dict()
        ),
        "projection_optimizer_state_dict": (
            main_projection_optimizer
            .state_dict()
        ),
        "script_optimizer_state_dict": (
            main_script_optimizer
            .state_dict()
        ),
        "input_dimension": int(
            DINO_S_FEATURE_DIM
        ),
        "projection_dimension": int(
            PROJECTION_DIM
        ),
        "projection_parameters": 55440,
        "script_head_parameters": 290,
        "projection_learning_rate": float(
            ADVERSARIAL_PROJECTION_LR
        ),
        "script_head_learning_rate": float(
            ADVERSARIAL_SCRIPT_HEAD_LR
        ),
        "weight_decay": float(
            ADVERSARIAL_WEIGHT_DECAY
        ),
        "cosine_margin": float(
            COSINE_MARGIN
        ),
        "grl_strength": float(
            FROZEN_GRL_STRENGTH
        ),
        "script_head_warmup_epochs": int(
            SCRIPT_HEAD_WARMUP_EPOCHS
        ),
        "script_head_warmup_source": (
            "fit-only competent warmed head"
        ),
        "script_optimizer_state_after_warmup_reused": False,
        "fit_writers": 145,
        "selection_writers": 36,
        "selection_checkpointing_used": False,
        "selection_evaluated_during_training": False,
        "early_stopping_used": False,
        "epoch_extension_used": False,
        "monitor_used": False,
        "validation_used": False,
        "official_test_used": False,
    },
    ADVERSARIAL_CHECKPOINT_PATH,
)


main_adversarial_history_df.to_csv(
    ADVERSARIAL_HISTORY_PATH,
    index=False,
)

main_adversarial_batch_df.to_csv(
    ADVERSARIAL_BATCH_HISTORY_PATH,
    index=False,
)


main_adversarial_model.eval()

with torch.no_grad():
    final_fit_embeddings = (
        main_adversarial_model
        .project(
            torch.from_numpy(
                fit_raw_features
            ).float()
        )
    )

    final_fit_script_logits = (
        main_adversarial_model
        .classify_script(
            final_fit_embeddings,
            use_grl=False,
        )
    )

    final_fit_script_probabilities = (
        torch.softmax(
            final_fit_script_logits,
            dim=1,
        )[
            :,
            1,
        ]
        .cpu()
        .numpy()
    )

    final_fit_script_predictions = (
        final_fit_script_logits
        .argmax(
            dim=1
        )
        .cpu()
        .numpy()
    )

    final_fit_attached_script_loss = float(
        F.cross_entropy(
            final_fit_script_logits,
            torch.from_numpy(
                fit_script_labels
            ).long(),
        )
        .item()
    )


final_fit_attached_script_auc = float(
    roc_auc_score(
        fit_script_labels,
        final_fit_script_probabilities,
    )
)

final_fit_attached_script_accuracy = float(
    accuracy_score(
        fit_script_labels,
        final_fit_script_predictions,
    )
)


first_epoch_row = (
    main_adversarial_history_df
    .iloc[
        0
    ]
)

final_epoch_row = (
    main_adversarial_history_df
    .iloc[
        -1
    ]
)


main_adversarial_training_summary = {
    "method": (
        "DINOv2-S script-adversarial projection"
    ),
    "training_seed": int(
        MAIN_ADVERSARIAL_SEED
    ),
    "fit_writers": 145,
    "fit_pages": 580,
    "epochs_completed": int(
        len(
            main_adversarial_history_df
        )
    ),
    "batches_per_epoch": 18,
    "total_training_batches": int(
        len(
            main_adversarial_batch_df
        )
    ),
    "script_head_optimizer_steps": int(
        len(
            main_adversarial_batch_df
        )
    ),
    "projection_optimizer_steps": int(
        len(
            main_adversarial_batch_df
        )
    ),
    "total_optimizer_steps": int(
        2
        * len(
            main_adversarial_batch_df
        )
    ),
    "pretraining_first_batch_writer_loss": float(
        pretraining_writer_loss
    ),
    "expected_notebook27_initial_writer_loss": float(
        expected_notebook27_initial_base_loss
    ),
    "pretraining_writer_loss_difference": float(
        pretraining_writer_loss_difference
    ),
    "pretraining_first_batch_script_loss": float(
        pretraining_first_batch_script_loss
    ),
    "epoch1_writer_base_loss": float(
        first_epoch_row[
            "writer_base_loss"
        ]
    ),
    "epoch10_writer_base_loss": float(
        final_epoch_row[
            "writer_base_loss"
        ]
    ),
    "writer_base_loss_change_epoch1_to_epoch10": float(
        final_epoch_row[
            "writer_base_loss"
        ]
        - first_epoch_row[
            "writer_base_loss"
        ]
    ),
    "epoch10_arabic_metric_loss": float(
        final_epoch_row[
            "arabic_metric_loss"
        ]
    ),
    "epoch10_cross_metric_loss": float(
        final_epoch_row[
            "cross_metric_loss"
        ]
    ),
    "epoch1_script_head_step_accuracy": float(
        first_epoch_row[
            "script_head_step_accuracy"
        ]
    ),
    "epoch10_script_head_step_accuracy": float(
        final_epoch_row[
            "script_head_step_accuracy"
        ]
    ),
    "epoch1_post_step_script_accuracy": float(
        first_epoch_row[
            "post_step_script_accuracy"
        ]
    ),
    "epoch10_post_step_script_accuracy": float(
        final_epoch_row[
            "post_step_script_accuracy"
        ]
    ),
    "final_fit_attached_script_auc": float(
        final_fit_attached_script_auc
    ),
    "final_fit_attached_script_accuracy": float(
        final_fit_attached_script_accuracy
    ),
    "final_fit_attached_script_loss": float(
        final_fit_attached_script_loss
    ),
    "projection_gradient_norm_mean_all_batches": float(
        main_adversarial_batch_df[
            "projection_gradient_norm_before_clip"
        ].mean()
    ),
    "projection_gradient_norm_max_all_batches": float(
        main_adversarial_batch_df[
            "projection_gradient_norm_before_clip"
        ].max()
    ),
    "projection_gradient_clipping_events_total": int(
        main_adversarial_batch_df[
            "projection_gradient_clipping_activated"
        ].sum()
    ),
    "script_head_gradient_norm_mean_all_batches": float(
        main_adversarial_batch_df[
            "script_head_gradient_norm"
        ].mean()
    ),
    "frozen_grl_strength": float(
        FROZEN_GRL_STRENGTH
    ),
    "checkpoint_path": str(
        ADVERSARIAL_CHECKPOINT_PATH
    ),
    "history_path": str(
        ADVERSARIAL_HISTORY_PATH
    ),
    "batch_history_path": str(
        ADVERSARIAL_BATCH_HISTORY_PATH
    ),
    "fixed_epoch10_training_completed": True,
    "selection_checkpointing_used": False,
    "selection_evaluated_during_training": False,
    "selection_evaluated_after_training": False,
    "attached_head_result_used_as_invariance_evidence": False,
    "independent_posthoc_probe_still_required": True,
    "early_stopping_used": False,
    "epoch_extension_used": False,
    "architecture_changed_after_protocol_freeze": False,
    "grl_strength_changed_after_protocol_freeze": False,
    "monitor_used": False,
    "validation_used": False,
    "official_test_used": False,
}


with open(
    REPORT_DIR
    / "dinov2s_script_adversarial_seed42_training_summary.json",
    "w",
) as file:
    json.dump(
        main_adversarial_training_summary,
        file,
        indent=2,
    )


print(
    "\nSeed-42 adversarial training summary:"
)

print(
    json.dumps(
        main_adversarial_training_summary,
        indent=2,
    )
)

print(
    "\nAdversarial epoch history:"
)

print(
    main_adversarial_history_df[
        [
            "epoch",
            "writer_base_loss",
            "arabic_metric_loss",
            "cross_metric_loss",
            "script_head_step_loss",
            "script_head_step_accuracy",
            "post_step_script_loss",
            "post_step_script_accuracy",
            "projection_gradient_norm_mean",
            "projection_gradient_clipping_events",
        ]
    ]
    .round(
        6
    )
    .to_string(
        index=False
    )
)


if (
    abs(
        pretraining_writer_loss_difference
    )
    > 1e-7
    or len(
        main_adversarial_history_df
    ) != 10
    or len(
        main_adversarial_batch_df
    ) != 180
    or not (
        main_adversarial_history_df[
            "batches"
        ]
        == 18
    ).all()
    or not (
        main_adversarial_history_df[
            "writers_seen"
        ]
        == 145
    ).all()
    or not np.isfinite(
        main_adversarial_history_df[
            [
                "writer_base_loss",
                "arabic_metric_loss",
                "cross_metric_loss",
                "script_head_step_loss",
                "script_head_step_accuracy",
                "post_step_script_loss",
                "post_step_script_accuracy",
                "projection_gradient_norm_mean",
            ]
        ].to_numpy()
    ).all()
    or not ADVERSARIAL_CHECKPOINT_PATH.exists()
):
    raise RuntimeError(
        "Seed-42 adversarial fixed training audit failed."
    )

dinov2s-script-adv | seed 42 | epoch 01 | writer 0.411313 | arabic 0.193672 | cross 0.217641 | head loss 0.903665 | head acc 0.6276 | post acc 0.6000 | proj grad 0.296292
dinov2s-script-adv | seed 42 | epoch 02 | writer 0.368996 | arabic 0.166427 | cross 0.202568 | head loss 1.426680 | head acc 0.5000 | post acc 0.5000 | proj grad 0.273674
dinov2s-script-adv | seed 42 | epoch 03 | writer 0.328956 | arabic 0.141706 | cross 0.187250 | head loss 1.303522 | head acc 0.5000 | post acc 0.5000 | proj grad 0.275473
dinov2s-script-adv | seed 42 | epoch 04 | writer 0.315674 | arabic 0.132683 | cross 0.182991 | head loss 1.198216 | head acc 0.5000 | post acc 0.5000 | proj grad 0.266159
dinov2s-script-adv | seed 42 | epoch 05 | writer 0.301912 | arabic 0.124286 | cross 0.177625 | head loss 1.093387 | head acc 0.5000 | post acc 0.5000 | proj grad 0.270870
dinov2s-script-adv | seed 42 | epoch 06 | writer 0.289722 | arabic 0.118682 | cross 0.171040 | head loss 1.002924 | head acc 0.5000 | post acc 0.

In [12]:
main_adversarial_model.eval()

with torch.no_grad():
    adversarial_fit_embeddings = (
        main_adversarial_model
        .project(
            torch.from_numpy(
                fit_raw_features
            ).float()
        )
        .cpu()
        .numpy()
    )

    adversarial_selection_embeddings = (
        main_adversarial_model
        .project(
            torch.from_numpy(
                selection_raw_features
            ).float()
        )
        .cpu()
        .numpy()
    )

    attached_selection_logits = (
        main_adversarial_model
        .classify_script(
            torch.from_numpy(
                adversarial_selection_embeddings
            ).float(),
            use_grl=False,
        )
    )

    attached_selection_probabilities = (
        torch.softmax(
            attached_selection_logits,
            dim=1,
        )[
            :,
            1,
        ]
        .cpu()
        .numpy()
    )

    attached_selection_predictions = (
        attached_selection_logits
        .argmax(
            dim=1
        )
        .cpu()
        .numpy()
    )


adversarial_selection_embedding_map = {
    filename: embedding
    for filename, embedding in zip(
        selection_rows[
            "filename"
        ].astype(str),
        adversarial_selection_embeddings,
    )
}


adversarial_writer_evaluation = (
    evaluate_selection_embedding_map(
        adversarial_selection_embedding_map
    )
)


adversarial_condition_df = (
    adversarial_writer_evaluation[
        "condition_df"
    ]
    .copy()
)


adversarial_condition_df[
    "baseline_auc"
] = (
    adversarial_condition_df[
        "condition"
    ]
    .map(
        baseline_condition_auc_map
    )
)


adversarial_condition_df[
    "auc_change_vs_baseline"
] = (
    adversarial_condition_df[
        "auc"
    ]
    - adversarial_condition_df[
        "baseline_auc"
    ]
)


adversarial_condition_df[
    "auc_drop_vs_baseline"
] = (
    adversarial_condition_df[
        "baseline_auc"
    ]
    - adversarial_condition_df[
        "auc"
    ]
)


script_probe = Pipeline(
    [
        (
            "scaler",
            StandardScaler(),
        ),
        (
            "classifier",
            LogisticRegression(
                C=1.0,
                solver="lbfgs",
                max_iter=5000,
                random_state=42,
            ),
        ),
    ]
)


script_probe.fit(
    adversarial_fit_embeddings,
    fit_script_labels,
)


adversarial_fit_script_probabilities = (
    script_probe.predict_proba(
        adversarial_fit_embeddings
    )[
        :,
        1,
    ]
)

adversarial_selection_script_probabilities = (
    script_probe.predict_proba(
        adversarial_selection_embeddings
    )[
        :,
        1,
    ]
)


adversarial_fit_script_predictions = (
    script_probe.predict(
        adversarial_fit_embeddings
    )
)

adversarial_selection_script_predictions = (
    script_probe.predict(
        adversarial_selection_embeddings
    )
)


posthoc_fit_script_auc = float(
    roc_auc_score(
        fit_script_labels,
        adversarial_fit_script_probabilities,
    )
)

posthoc_fit_script_accuracy = float(
    accuracy_score(
        fit_script_labels,
        adversarial_fit_script_predictions,
    )
)

posthoc_fit_script_balanced_accuracy = float(
    balanced_accuracy_score(
        fit_script_labels,
        adversarial_fit_script_predictions,
    )
)


posthoc_selection_script_auc = float(
    roc_auc_score(
        selection_script_labels,
        adversarial_selection_script_probabilities,
    )
)

posthoc_selection_script_accuracy = float(
    accuracy_score(
        selection_script_labels,
        adversarial_selection_script_predictions,
    )
)

posthoc_selection_script_balanced_accuracy = float(
    balanced_accuracy_score(
        selection_script_labels,
        adversarial_selection_script_predictions,
    )
)

posthoc_selection_script_confusion = (
    confusion_matrix(
        selection_script_labels,
        adversarial_selection_script_predictions,
        labels=[
            0,
            1,
        ],
    )
)


attached_selection_script_auc = float(
    roc_auc_score(
        selection_script_labels,
        attached_selection_probabilities,
    )
)

attached_selection_script_accuracy = float(
    accuracy_score(
        selection_script_labels,
        attached_selection_predictions,
    )
)


writer_macro_auc = float(
    adversarial_writer_evaluation[
        "cross_macro_auc"
    ]
)

writer_macro_eer = float(
    adversarial_writer_evaluation[
        "cross_macro_eer"
    ]
)

writer_pooled_auc = float(
    adversarial_writer_evaluation[
        "pooled_auc"
    ]
)

writer_pooled_eer = float(
    adversarial_writer_evaluation[
        "pooled_eer"
    ]
)


writer_macro_auc_change = float(
    writer_macro_auc
    - BASELINE_SEED42_MACRO_AUC
)

writer_macro_auc_drop = float(
    BASELINE_SEED42_MACRO_AUC
    - writer_macro_auc
)


script_probe_auc_reduction = float(
    BASELINE_SCRIPT_PROBE_AUC
    - posthoc_selection_script_auc
)


maximum_condition_auc_drop = float(
    adversarial_condition_df[
        "auc_drop_vs_baseline"
    ]
    .max()
)


writer_macro_guardrail_passed = bool(
    writer_macro_auc_drop
    <= WRITER_MACRO_AUC_MAXIMUM_DROP
)


condition_guardrail_passed = bool(
    maximum_condition_auc_drop
    <= WRITER_CONDITION_AUC_MAXIMUM_DROP
)


script_suppression_target_passed = bool(
    script_probe_auc_reduction
    >= SCRIPT_AUC_MINIMUM_REDUCTION
)


pareto_support_passed = bool(
    script_suppression_target_passed
    and writer_macro_guardrail_passed
    and condition_guardrail_passed
)


if pareto_support_passed:
    decision = (
        "pareto_supported"
    )
elif (
    script_suppression_target_passed
    and (
        not writer_macro_guardrail_passed
        or not condition_guardrail_passed
    )
):
    decision = (
        "invariance_performance_tradeoff"
    )
else:
    decision = (
        "insufficient_script_suppression"
    )


script_probe_classifier = (
    script_probe.named_steps[
        "classifier"
    ]
)


selection_result_summary = {
    "method": (
        "DINOv2-S script-adversarial projection"
    ),
    "training_seed": 42,
    "checkpoint_epoch": 10,
    "writer_selection_cross_macro_auc": float(
        writer_macro_auc
    ),
    "writer_selection_cross_macro_eer": float(
        writer_macro_eer
    ),
    "writer_selection_pooled_auc": float(
        writer_pooled_auc
    ),
    "writer_selection_pooled_eer": float(
        writer_pooled_eer
    ),
    "baseline_writer_cross_macro_auc": float(
        BASELINE_SEED42_MACRO_AUC
    ),
    "writer_cross_macro_auc_change_vs_baseline": float(
        writer_macro_auc_change
    ),
    "writer_cross_macro_auc_drop_vs_baseline": float(
        writer_macro_auc_drop
    ),
    "posthoc_script_probe_fit_auc": float(
        posthoc_fit_script_auc
    ),
    "posthoc_script_probe_fit_accuracy": float(
        posthoc_fit_script_accuracy
    ),
    "posthoc_script_probe_fit_balanced_accuracy": float(
        posthoc_fit_script_balanced_accuracy
    ),
    "posthoc_script_probe_selection_auc": float(
        posthoc_selection_script_auc
    ),
    "posthoc_script_probe_selection_accuracy": float(
        posthoc_selection_script_accuracy
    ),
    "posthoc_script_probe_selection_balanced_accuracy": float(
        posthoc_selection_script_balanced_accuracy
    ),
    "posthoc_script_probe_selection_confusion_matrix": (
        posthoc_selection_script_confusion
        .astype(int)
        .tolist()
    ),
    "posthoc_script_probe_iterations": int(
        script_probe_classifier
        .n_iter_[
            0
        ]
    ),
    "baseline_projected_script_probe_auc": float(
        BASELINE_SCRIPT_PROBE_AUC
    ),
    "script_probe_auc_reduction_vs_baseline": float(
        script_probe_auc_reduction
    ),
    "attached_head_selection_auc_diagnostic_only": float(
        attached_selection_script_auc
    ),
    "attached_head_selection_accuracy_diagnostic_only": float(
        attached_selection_script_accuracy
    ),
    "minimum_required_script_auc_reduction": float(
        SCRIPT_AUC_MINIMUM_REDUCTION
    ),
    "maximum_allowed_writer_macro_auc_drop": float(
        WRITER_MACRO_AUC_MAXIMUM_DROP
    ),
    "maximum_observed_condition_auc_drop": float(
        maximum_condition_auc_drop
    ),
    "maximum_allowed_condition_auc_drop": float(
        WRITER_CONDITION_AUC_MAXIMUM_DROP
    ),
    "script_suppression_target_passed": bool(
        script_suppression_target_passed
    ),
    "writer_macro_guardrail_passed": bool(
        writer_macro_guardrail_passed
    ),
    "condition_guardrail_passed": bool(
        condition_guardrail_passed
    ),
    "pareto_support_passed": bool(
        pareto_support_passed
    ),
    "decision": (
        decision
    ),
    "attached_head_used_as_invariance_evidence": False,
    "independent_posthoc_probe_used": True,
    "selection_used_for_checkpoint_selection": False,
    "selection_used_to_choose_grl_strength": False,
    "epoch_extension_allowed_after_result": False,
    "monitor_used": False,
    "validation_used": False,
    "official_test_used": False,
}


adversarial_condition_df.to_csv(
    REPORT_DIR
    / "dinov2s_script_adversarial_seed42_selection_conditions.csv",
    index=False,
)


with open(
    REPORT_DIR
    / "dinov2s_script_adversarial_seed42_selection_summary.json",
    "w",
) as file:
    json.dump(
        selection_result_summary,
        file,
        indent=2,
    )


np.savez_compressed(
    REPORT_DIR
    / "dinov2s_script_adversarial_seed42_selection_embeddings.npz",
    fit_embeddings=(
        adversarial_fit_embeddings
    ),
    selection_embeddings=(
        adversarial_selection_embeddings
    ),
    fit_script_labels=(
        fit_script_labels
    ),
    selection_script_labels=(
        selection_script_labels
    ),
    fit_filenames=np.asarray(
        fit_rows[
            "filename"
        ]
        .astype(str)
        .tolist(),
        dtype=str,
    ),
    selection_filenames=np.asarray(
        selection_rows[
            "filename"
        ]
        .astype(str)
        .tolist(),
        dtype=str,
    ),
)


print(
    json.dumps(
        selection_result_summary,
        indent=2,
    )
)

print(
    "\nWriter-verification condition comparison:"
)

print(
    adversarial_condition_df[
        [
            "condition",
            "baseline_auc",
            "auc",
            "auc_change_vs_baseline",
            "auc_drop_vs_baseline",
            "eer",
        ]
    ]
    .round(
        6
    )
    .to_string(
        index=False
    )
)


if (
    adversarial_fit_embeddings.shape
    != (
        580,
        144,
    )
    or adversarial_selection_embeddings.shape
    != (
        144,
        144,
    )
    or len(
        adversarial_condition_df
    ) != 4
    or not np.isfinite(
        [
            writer_macro_auc,
            writer_macro_eer,
            writer_pooled_auc,
            writer_pooled_eer,
            posthoc_fit_script_auc,
            posthoc_selection_script_auc,
            attached_selection_script_auc,
        ]
    ).all()
):
    raise RuntimeError(
        "Seed-42 adversarial selection evaluation failed."
    )

{
  "method": "DINOv2-S script-adversarial projection",
  "training_seed": 42,
  "checkpoint_epoch": 10,
  "writer_selection_cross_macro_auc": 0.8254464285714287,
  "writer_selection_cross_macro_eer": 0.24682539682539684,
  "writer_selection_pooled_auc": 0.8258060515873016,
  "writer_selection_pooled_eer": 0.23809523809523808,
  "baseline_writer_cross_macro_auc": 0.8374834656084656,
  "writer_cross_macro_auc_change_vs_baseline": -0.012037037037036957,
  "writer_cross_macro_auc_drop_vs_baseline": 0.012037037037036957,
  "posthoc_script_probe_fit_auc": 0.9879429250891796,
  "posthoc_script_probe_fit_accuracy": 0.9431034482758621,
  "posthoc_script_probe_fit_balanced_accuracy": 0.943103448275862,
  "posthoc_script_probe_selection_auc": 0.9056712962962964,
  "posthoc_script_probe_selection_accuracy": 0.8194444444444444,
  "posthoc_script_probe_selection_balanced_accuracy": 0.8194444444444444,
  "posthoc_script_probe_selection_confusion_matrix": [
    [
      60,
      12
    ],
    [
     

In [13]:
adversarial_fit_text_labels = np.where(
    fit_rows[
        "page_id"
    ]
    .astype(int)
    .isin(
        [
            2,
            4,
        ]
    ),
    1,
    0,
).astype(
    np.int64
)

adversarial_selection_text_labels = np.where(
    selection_rows[
        "page_id"
    ]
    .astype(int)
    .isin(
        [
            2,
            4,
        ]
    ),
    1,
    0,
).astype(
    np.int64
)


text_condition_probe = Pipeline(
    [
        (
            "scaler",
            StandardScaler(),
        ),
        (
            "classifier",
            LogisticRegression(
                C=1.0,
                solver="lbfgs",
                max_iter=5000,
                random_state=42,
            ),
        ),
    ]
)


text_condition_probe.fit(
    adversarial_fit_embeddings,
    adversarial_fit_text_labels,
)


text_fit_probabilities = (
    text_condition_probe.predict_proba(
        adversarial_fit_embeddings
    )[
        :,
        1,
    ]
)

text_selection_probabilities = (
    text_condition_probe.predict_proba(
        adversarial_selection_embeddings
    )[
        :,
        1,
    ]
)


text_fit_predictions = (
    text_condition_probe.predict(
        adversarial_fit_embeddings
    )
)

text_selection_predictions = (
    text_condition_probe.predict(
        adversarial_selection_embeddings
    )
)


adversarial_text_fit_auc = float(
    roc_auc_score(
        adversarial_fit_text_labels,
        text_fit_probabilities,
    )
)

adversarial_text_fit_accuracy = float(
    accuracy_score(
        adversarial_fit_text_labels,
        text_fit_predictions,
    )
)

adversarial_text_selection_auc = float(
    roc_auc_score(
        adversarial_selection_text_labels,
        text_selection_probabilities,
    )
)

adversarial_text_selection_accuracy = float(
    accuracy_score(
        adversarial_selection_text_labels,
        text_selection_predictions,
    )
)

adversarial_text_selection_balanced_accuracy = float(
    balanced_accuracy_score(
        adversarial_selection_text_labels,
        text_selection_predictions,
    )
)

adversarial_text_selection_confusion = (
    confusion_matrix(
        adversarial_selection_text_labels,
        text_selection_predictions,
        labels=[
            0,
            1,
        ],
    )
)


text_auc_change_vs_baseline = float(
    adversarial_text_selection_auc
    - BASELINE_TEXT_PROBE_AUC
)


all_condition_changes_negative = bool(
    (
        adversarial_condition_df[
            "auc_change_vs_baseline"
        ]
        < 0.0
    ).all()
)


multiseed_justified = bool(
    selection_result_summary[
        "pareto_support_passed"
    ]
)


final_notebook29_verdict = {
    "notebook": 29,
    "method": (
        "DINOv2-S script-adversarial projection"
    ),
    "seed42_writer_macro_auc": float(
        writer_macro_auc
    ),
    "baseline_writer_macro_auc": float(
        BASELINE_SEED42_MACRO_AUC
    ),
    "writer_macro_auc_change": float(
        writer_macro_auc_change
    ),
    "seed42_writer_pooled_auc": float(
        writer_pooled_auc
    ),
    "seed42_writer_pooled_eer": float(
        writer_pooled_eer
    ),
    "all_four_writer_conditions_declined": bool(
        all_condition_changes_negative
    ),
    "maximum_condition_auc_drop": float(
        maximum_condition_auc_drop
    ),
    "baseline_script_probe_auc": float(
        BASELINE_SCRIPT_PROBE_AUC
    ),
    "adversarial_posthoc_script_probe_auc": float(
        posthoc_selection_script_auc
    ),
    "script_probe_auc_change": float(
        posthoc_selection_script_auc
        - BASELINE_SCRIPT_PROBE_AUC
    ),
    "attached_head_selection_auc_diagnostic_only": float(
        attached_selection_script_auc
    ),
    "attached_head_selection_accuracy_diagnostic_only": float(
        attached_selection_script_accuracy
    ),
    "attached_head_and_posthoc_probe_disagree": bool(
        attached_selection_script_auc
        < 0.5
        and posthoc_selection_script_auc
        > BASELINE_SCRIPT_PROBE_AUC
    ),
    "baseline_text_condition_probe_auc": float(
        BASELINE_TEXT_PROBE_AUC
    ),
    "adversarial_text_condition_probe_fit_auc": float(
        adversarial_text_fit_auc
    ),
    "adversarial_text_condition_probe_fit_accuracy": float(
        adversarial_text_fit_accuracy
    ),
    "adversarial_text_condition_probe_selection_auc": float(
        adversarial_text_selection_auc
    ),
    "adversarial_text_condition_probe_selection_accuracy": float(
        adversarial_text_selection_accuracy
    ),
    "adversarial_text_condition_probe_selection_balanced_accuracy": float(
        adversarial_text_selection_balanced_accuracy
    ),
    "adversarial_text_condition_probe_selection_confusion_matrix": (
        adversarial_text_selection_confusion
        .astype(int)
        .tolist()
    ),
    "text_condition_probe_auc_change_vs_baseline": float(
        text_auc_change_vs_baseline
    ),
    "script_suppression_target_passed": bool(
        script_suppression_target_passed
    ),
    "writer_macro_guardrail_passed": bool(
        writer_macro_guardrail_passed
    ),
    "condition_guardrail_passed": bool(
        condition_guardrail_passed
    ),
    "pareto_support_passed": bool(
        pareto_support_passed
    ),
    "seed42_decision": (
        selection_result_summary[
            "decision"
        ]
    ),
    "method_dominates_notebook27_baseline": False,
    "notebook27_baseline_remains_preferred": True,
    "multiseed_justified_by_frozen_protocol": bool(
        multiseed_justified
    ),
    "run_additional_seeds": False,
    "tune_grl_strength_after_result": False,
    "extend_epoch_budget_after_result": False,
    "change_adversary_architecture_after_result": False,
    "reuse_monitor_for_method_development": False,
    "scientific_interpretation": (
        "The attached adversarial head was strongly confused, "
        "but an independent writer-disjoint linear probe recovered "
        "more script information than from the standard projection. "
        "Writer verification also declined across all four conditions. "
        "Therefore the attached-head confusion is not evidence of "
        "script invariance, and this explicit adversarial method is "
        "not supported as a replacement for the Notebook 27 baseline."
    ),
    "selection_stage_closed": True,
    "monitor_used": False,
    "validation_used": False,
    "official_test_used": False,
}


with open(
    REPORT_DIR
    / "dinov2s_script_adversarial_final_verdict.json",
    "w",
) as file:
    json.dump(
        final_notebook29_verdict,
        file,
        indent=2,
    )


print(
    json.dumps(
        final_notebook29_verdict,
        indent=2,
    )
)


if (
    final_notebook29_verdict[
        "run_additional_seeds"
    ]
    or final_notebook29_verdict[
        "tune_grl_strength_after_result"
    ]
    or final_notebook29_verdict[
        "extend_epoch_budget_after_result"
    ]
    or final_notebook29_verdict[
        "reuse_monitor_for_method_development"
    ]
):
    raise RuntimeError(
        "Frozen negative-result protocol was violated."
    )

{
  "notebook": 29,
  "method": "DINOv2-S script-adversarial projection",
  "seed42_writer_macro_auc": 0.8254464285714287,
  "baseline_writer_macro_auc": 0.8374834656084656,
  "writer_macro_auc_change": -0.012037037037036957,
  "seed42_writer_pooled_auc": 0.8258060515873016,
  "seed42_writer_pooled_eer": 0.23809523809523808,
  "all_four_writer_conditions_declined": true,
  "maximum_condition_auc_drop": 0.021759259259259256,
  "baseline_script_probe_auc": 0.757908950617284,
  "adversarial_posthoc_script_probe_auc": 0.9056712962962964,
  "script_probe_auc_change": 0.14776234567901236,
  "attached_head_selection_auc_diagnostic_only": 0.3385416666666667,
  "attached_head_selection_accuracy_diagnostic_only": 0.5,
  "attached_head_and_posthoc_probe_disagree": true,
  "baseline_text_condition_probe_auc": 0.7924382716049383,
  "adversarial_text_condition_probe_fit_auc": 0.9358501783590963,
  "adversarial_text_condition_probe_fit_accuracy": 0.8603448275862069,
  "adversarial_text_condition_prob

## Final Summary

Notebook 29 tested whether explicit script-adversarial training could improve the writer-discrimination versus script-invariance trade-off of the DINOv2-S projection student established in Notebook 27.

The experiment was intentionally controlled.

The frozen DINOv2-S 384-D representation and the 384→144 writer projection architecture were unchanged. The writer metric objective, training schedule, learning rate, weight decay, cosine margin, and 10-epoch budget remained matched to the established projection baseline.

The only new optimization factor was an explicit Arabic-versus-English adversarial branch attached to the normalized 144-D writer embedding through gradient reversal.

### Adversarial-System Audit

Before the main experiment, several safeguards were verified using fit-only data.

The linear script head was first trained with the writer projection frozen and successfully learned the script task:

- Fit script ROC-AUC: **0.93424**
- Fit script accuracy: **0.85345**
- Projection parameter change during warm-up: **0**

Therefore, the attached classifier was demonstrably capable of learning script information before adversarial training.

The gradient-reversal implementation was also verified directly.

For the projection parameters:

**direct script gradient versus GRL script gradient cosine ≈ -1.0**

while for the script-head parameters:

**direct versus GRL gradient cosine ≈ +1.0**

Thus, gradient reversal changed only the projection-side direction as intended.

The adversarial strength was not selected using writer-verification outcomes on the selection set.

Using six fit-only batches, the median script-adversarial projection-gradient magnitude at GRL strength 1 was approximately **1.767×** the writer-gradient magnitude.

The GRL coefficient was therefore frozen to:

**λ = 0.11316**

so that the median adversarial contribution at initialization was approximately **20%** of the writer-gradient magnitude.

No selection result was used to tune this coefficient.

### Training Behavior

The frozen seed-42 protocol was trained for exactly 10 epochs using alternating optimization:

1. one script-head update with the projection frozen,
2. one writer-plus-GRL projection update with the script head frozen.

Training was numerically stable.

The mean writer loss decreased from approximately:

**0.41131 → 0.25631**

between epochs 1 and 10.

No projection-gradient clipping events occurred.

However, the attached script head became strongly confused during adversarial training, reaching approximately chance-level accuracy.

At the final fit representation, the attached head had:

- Accuracy: **0.50000**
- ROC-AUC: **0.19058**

This attached-head result was deliberately not treated as evidence of script invariance.

### Independent Script Accessibility

The decisive nuisance evaluation used a newly fitted independent linear probe on the frozen epoch-10 representation, following the same writer-disjoint protocol as Notebook 28.

The standard Notebook 27 projection had:

**Script probe ROC-AUC = 0.75791**

The adversarial projection produced:

**Script probe ROC-AUC = 0.90567**

Therefore:

**Δ script probe AUC = +0.14776**

Script information did not become less linearly accessible.

It became substantially more accessible to an independent probe despite the attached adversarial head being highly confused.

The independent probe achieved:

- Selection accuracy: **0.81944**
- Selection balanced accuracy: **0.81944**

This directly demonstrates that confusion of the attached adversarial classifier is not sufficient evidence that script information has been removed from the representation.

### Writer Verification

The standard seed-42 Notebook 27 projection achieved:

**Cross-script macro AUC = 0.83748**

The script-adversarial projection achieved:

**Cross-script macro AUC = 0.82545**

Therefore:

**Δ macro AUC = -0.01204**

The pooled adversarial result was:

- Pooled AUC: **0.82581**
- Pooled EER: **0.23810**

All four cross-script conditions declined relative to the standard projection:

| Condition | Baseline AUC | Adversarial AUC | Change |
|---|---:|---:|---:|
| cross_same_same | 0.86905 | 0.85813 | -0.01091 |
| cross_same_variable | 0.84184 | 0.83739 | -0.00445 |
| cross_variable_same | 0.77822 | 0.76720 | -0.01102 |
| cross_variable_variable | 0.86082 | 0.83907 | -0.02176 |

The largest condition decline was approximately:

**-0.02176 AUC**

### Secondary Text-Condition Diagnostic

Text condition was never used as an adversarial training target.

It remained a diagnostic variable.

The Notebook 28 standard projection had:

**Text-condition probe ROC-AUC = 0.79244**

After script-adversarial training:

**Text-condition probe ROC-AUC = 0.81385**

Therefore:

**Δ text-condition probe AUC = +0.02141**

Explicit script adversarial training therefore did not produce broader nuisance suppression.

### Predefined Decision Criteria

Before observing the selection result, Pareto support required:

- script-probe AUC reduction of at least **0.05**,
- writer macro-AUC drop no greater than **0.005**,
- no individual cross-script condition drop greater than **0.02**.

The experiment failed all three criteria.

The script-probe AUC increased rather than decreased.

The writer macro-AUC declined by approximately **0.01204**.

The maximum condition-level decline was approximately **0.02176**.

Therefore:

**Pareto support: Not achieved**

### Scientific Interpretation

This experiment does not support the explicit script-adversarial projection as a replacement for the standard DINOv2-S writer projection.

The result is stronger than a simple invariance-performance trade-off.

A trade-off would require reduced script accessibility at the cost of some writer discrimination.

Instead, the adversarial model simultaneously produced:

- lower writer-verification performance,
- higher independent script accessibility,
- slightly higher text-condition accessibility.

The attached adversarial head was successfully confused, but the underlying representation remained strongly script-decodable by a fresh independent classifier.

This suggests that the projection learned a representation that defeated the particular attached classifier without eliminating the broader linearly accessible script structure.

Therefore:

> Attached-head confusion must not be interpreted as representation-level invariance without an independent post-hoc accessibility test.

### Research Decision

The Notebook 27 DINOv2-S projection remains the preferred student representation.

No additional adversarial seeds will be run because the predetermined seed-42 support criterion was not satisfied.

No post-hoc GRL-strength tuning, epoch extension, adversary redesign, or monitor reuse will be performed to rescue this result.

The negative result is retained because it provides a controlled and informative finding:

> The standard writer metric objective naturally reduced nuisance accessibility while preserving strong writer verification, whereas the tested explicit script-adversarial objective disrupted that favorable representation geometry and failed to provide genuine script suppression.

The selection stage for this method is now closed.

- Monitor used for Notebook 29 development: **No**
- Validation used: **No**
- Official test used: **No**